In [1]:
# ==================================================================================================
# CISLR — L20A
#
# FULL CISLR SUPERVISED DATASET AUDIT
# + DETERMINISTIC STRATIFIED 70/30 CANDIDATE SPLITS
#
# PURPOSE
# -------
# Build scientifically valid candidate supervised classification datasets
# from the full CISLR collection.
#
# IMPORTANT
# ---------
# This stage performs:
#
#     metadata audit
#     video-path resolution
#     class-support analysis
#     supervised eligibility analysis
#     deterministic 70/30 stratified candidate split generation
#
# This stage DOES NOT:
#
#     train a model
#     evaluate a model
#     choose a support threshold based on accuracy
#     use L19 test predictions
#     move validation videos into training
#
# WHY?
# ----
# CISLR is extremely long-tailed.
#
# A gloss represented by only one video cannot be placed in both:
#
#     TRAIN
#     VALIDATION
#
# Therefore:
#
#     "all ~7,000 videos"
#
# cannot automatically become a normal supervised 4,000+ class
# train/validation classification task.
#
# L20A measures exactly what is possible.
#
# ==================================================================================================


# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

import os
import cv2
import json
import math
import random
import hashlib
import platform

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# ==================================================================================================
# 2. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)


DATASET_CSV = Path(
    "/home/dipshikha/Music/cao/dataset.csv"
)


VIDEO_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_v1.5-a_videos"
)


# --------------------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------------------

L20A_ROOT = (
    ROOT /
    "audit/l20a_full_cislr_dataset_audit"
)


REPORT_DIR = (
    L20A_ROOT /
    "reports"
)


MANIFEST_DIR = (
    L20A_ROOT /
    "manifests"
)


SPLIT_DIR = (
    L20A_ROOT /
    "candidate_splits"
)


CONFIG_DIR = (
    L20A_ROOT /
    "config"
)


HASH_DIR = (
    L20A_ROOT /
    "hashes"
)


for directory in [

    L20A_ROOT,
    REPORT_DIR,
    MANIFEST_DIR,
    SPLIT_DIR,
    CONFIG_DIR,
    HASH_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 3. OUTPUT FILES
# ==================================================================================================

CLEAN_MANIFEST_PATH = (
    MANIFEST_DIR /
    "l20a_clean_full_cislr_manifest.csv"
)


INVALID_MANIFEST_PATH = (
    MANIFEST_DIR /
    "l20a_invalid_metadata_rows.csv"
)


SUPPORT_TABLE_PATH = (
    REPORT_DIR /
    "l20a_class_support_table.csv"
)


THRESHOLD_SUMMARY_PATH = (
    REPORT_DIR /
    "l20a_support_threshold_summary.csv"
)


SUMMARY_PATH = (
    REPORT_DIR /
    "l20a_summary.json"
)


HASH_MANIFEST_PATH = (
    HASH_DIR /
    "l20a_sha256_manifest.json"
)


FREEZE_PATH = (
    L20A_ROOT /
    "L20A_MASTER_FREEZE.json"
)


# ==================================================================================================
# 4. CONFIGURATION
# ==================================================================================================

TRAIN_FRACTION = 0.70

VALIDATION_FRACTION = 0.30


SUPPORT_THRESHOLDS = [

    2,
    3,
    4,
    5,
    6,
    8,
    10

]


# ==================================================================================================
# 5. HELPERS
# ==================================================================================================

def normalize_uid(value):

    uid = str(
        value
    ).strip()


    if uid.lower().endswith(
        ".mp4"
    ):

        uid = uid[:-4]


    return uid


def normalize_gloss(value):

    if pd.isna(
        value
    ):

        return ""


    return str(
        value
    ).strip().lower()


def sha256_file(path):

    h = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )


            if not block:

                break


            h.update(
                block
            )


    return h.hexdigest()


def save_json(
    obj,
    path
):

    with open(
        path,
        "w"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )


# ==================================================================================================
# 6. HEADER
# ==================================================================================================

print(
    "=" * 150
)

print(
    "CISLR — L20A"
)

print(
    "FULL CISLR SUPERVISED DATASET AUDIT + STRATIFIED 70/30 CANDIDATE SPLITS"
)

print(
    "=" * 150
)

print()


print(
    "Python:",
    platform.python_version()
)


print(
    "Pandas:",
    pd.__version__
)


print(
    "OpenCV:",
    cv2.__version__
)


print(
    "Seed:",
    SEED
)


print(
    "Train fraction:",
    TRAIN_FRACTION
)


print(
    "Validation fraction:",
    VALIDATION_FRACTION
)


print()


# ==================================================================================================
# 7. REQUIRED INPUT CHECK
# ==================================================================================================

print(
    "=" * 150
)

print(
    "1. REQUIRED INPUT CHECK"
)

print(
    "=" * 150
)


required = {

    "dataset.csv":
        DATASET_CSV,

    "CISLR video directory":
        VIDEO_ROOT

}


for name, path in (
    required.items()
):

    exists = path.exists()


    print(
        f"{name:30s}: "
        f"{'PASS' if exists else 'FAIL'}"
    )


    if not exists:

        raise FileNotFoundError(
            path
        )


print()

print(
    "[PASS] Required CISLR inputs found."
)


# ==================================================================================================
# 8. LOAD FULL DATASET
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "2. LOAD FULL CISLR METADATA"
)

print(
    "=" * 150
)


df = pd.read_csv(
    DATASET_CSV
)


print(
    "Metadata rows:",
    len(
        df
    )
)


print(
    "Columns:",
    list(
        df.columns
    )
)


required_columns = {

    "uid",
    "gloss"

}


missing_columns = (
    required_columns
    -
    set(
        df.columns
    )
)


if missing_columns:

    raise RuntimeError(

        f"Missing columns: "
        f"{sorted(missing_columns)}"

    )


# ==================================================================================================
# 9. NORMALIZE METADATA
# ==================================================================================================

df = df.copy()


df[
    "uid"
] = (
    df[
        "uid"
    ]
    .apply(
        normalize_uid
    )
)


df[
    "gloss"
] = (
    df[
        "gloss"
    ]
    .apply(
        normalize_gloss
    )
)


# ==================================================================================================
# 10. METADATA VALIDITY
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "3. METADATA VALIDITY"
)

print(
    "=" * 150
)


df[
    "valid_uid"
] = (
    df[
        "uid"
    ]
    .astype(str)
    .str.len()
    >
    0
)


df[
    "valid_gloss"
] = (
    df[
        "gloss"
    ]
    .astype(str)
    .str.len()
    >
    0
)


# defensive textual invalid labels

invalid_gloss_strings = {

    "",
    "nan",
    "none",
    "null",
    "na",
    "n/a"

}


df[
    "valid_gloss"
] = (

    df[
        "valid_gloss"
    ]

    &

    ~df[
        "gloss"
    ].isin(
        invalid_gloss_strings
    )

)


df[
    "metadata_valid"
] = (

    df[
        "valid_uid"
    ]

    &

    df[
        "valid_gloss"
    ]

)


invalid_df = df[
    ~df[
        "metadata_valid"
    ]
].copy()


valid_df = df[
    df[
        "metadata_valid"
    ]
].copy()


invalid_df.to_csv(
    INVALID_MANIFEST_PATH,
    index=False
)


print(
    "Metadata total:",
    len(
        df
    )
)


print(
    "Metadata valid:",
    len(
        valid_df
    )
)


print(
    "Metadata invalid:",
    len(
        invalid_df
    )
)


# ==================================================================================================
# 11. DUPLICATE UID CHECK
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "4. UID INTEGRITY"
)

print(
    "=" * 150
)


duplicate_uid_mask = (
    valid_df[
        "uid"
    ].duplicated(
        keep=False
    )
)


duplicate_uid_rows = valid_df[
    duplicate_uid_mask
]


print(
    "Duplicate UID rows:",
    len(
        duplicate_uid_rows
    )
)


print(
    "Unique valid UIDs:",
    valid_df[
        "uid"
    ].nunique()
)


if len(
    duplicate_uid_rows
) > 0:

    duplicate_path = (
        REPORT_DIR /
        "l20a_duplicate_uid_rows.csv"
    )


    duplicate_uid_rows.to_csv(
        duplicate_path,
        index=False
    )


    print(
        "[WARN] Duplicate UIDs detected."
    )

    print(
        "Report:",
        duplicate_path
    )


# ==================================================================================================
# 12. BUILD VIDEO FILE INDEX
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "5. BUILD CISLR VIDEO INDEX"
)

print(
    "=" * 150
)


video_files = list(
    VIDEO_ROOT.rglob(
        "*.mp4"
    )
)


print(
    "MP4 files discovered:",
    len(
        video_files
    )
)


video_index = {}


duplicate_video_stems = []


for path in video_files:

    stem = path.stem


    if stem in video_index:

        duplicate_video_stems.append(
            stem
        )

    else:

        video_index[
            stem
        ] = path


print(
    "Unique MP4 stems:",
    len(
        video_index
    )
)


print(
    "Duplicate MP4 stems:",
    len(
        duplicate_video_stems
    )
)


# ==================================================================================================
# 13. RESOLVE VIDEO PATHS
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "6. RESOLVE METADATA → VIDEO"
)

print(
    "=" * 150
)


resolved_paths = []

video_found = []


for uid in valid_df[
    "uid"
]:

    path = video_index.get(
        uid
    )


    if path is None:

        resolved_paths.append(
            ""
        )

        video_found.append(
            False
        )

    else:

        resolved_paths.append(
            str(
                path
            )
        )

        video_found.append(
            True
        )


valid_df[
    "video_path"
] = resolved_paths


valid_df[
    "video_found"
] = video_found


found_count = int(
    valid_df[
        "video_found"
    ].sum()
)


missing_count = int(
    (
        ~valid_df[
            "video_found"
        ]
    ).sum()
)


print(
    "Videos resolved:",
    found_count
)


print(
    "Videos missing:",
    missing_count
)


# ==================================================================================================
# 14. CREATE CLEAN VIDEO-BACKED DATASET
# ==================================================================================================

clean_df = valid_df[
    valid_df[
        "video_found"
    ]
].copy()


clean_df = (
    clean_df
    .drop_duplicates(
        subset=[
            "uid"
        ],
        keep="first"
    )
    .reset_index(
        drop=True
    )
)


clean_df.to_csv(
    CLEAN_MANIFEST_PATH,
    index=False
)


print()

print(
    "Clean video-backed samples:",
    len(
        clean_df
    )
)


print(
    "Clean unique glosses:",
    clean_df[
        "gloss"
    ].nunique()
)


# ==================================================================================================
# 15. CLASS SUPPORT
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "7. FULL CLASS-SUPPORT DISTRIBUTION"
)

print(
    "=" * 150
)


support = (
    clean_df
    .groupby(
        "gloss"
    )
    .size()
    .reset_index(
        name="support"
    )
)


support = (
    support
    .sort_values(
        [
            "support",
            "gloss"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


support[
    "class_rank"
] = (
    np.arange(
        len(
            support
        )
    )
    +
    1
)


support.to_csv(
    SUPPORT_TABLE_PATH,
    index=False
)


print(
    "Unique classes:",
    len(
        support
    )
)


print()


for threshold in [

    1,
    2,
    3,
    4,
    5,
    6,
    8,
    10

]:

    n_classes = int(
        (
            support[
                "support"
            ]
            >=
            threshold
        ).sum()
    )


    n_videos = int(

        support.loc[

            support[
                "support"
            ]
            >=
            threshold,

            "support"

        ].sum()

    )


    print(

        f"support >= {threshold:2d}: "
        f"{n_classes:5d} classes | "
        f"{n_videos:5d} videos"

    )


# ==================================================================================================
# 16. STRATIFIED SPLIT FUNCTION
# ==================================================================================================

def create_stratified_split(
    source_df,
    seed,
    train_fraction=0.70
):

    rng = np.random.default_rng(
        seed
    )


    train_parts = []

    val_parts = []


    for gloss, group in (
        source_df
        .groupby(
            "gloss",
            sort=True
        )
    ):

        group = (
            group
            .copy()
            .reset_index(
                drop=True
            )
        )


        n = len(
            group
        )


        if n < 2:

            raise RuntimeError(

                f"Cannot stratify class "
                f"{gloss} with n={n}"

            )


        permutation = rng.permutation(
            n
        )


        # ------------------------------------------------------------------------------------------
        # 70/30 split while guaranteeing:
        #
        # >=1 training
        # >=1 validation
        # ------------------------------------------------------------------------------------------

        n_train = int(
            math.floor(
                n *
                train_fraction
            )
        )


        n_train = max(
            1,
            n_train
        )


        n_train = min(
            n_train,
            n - 1
        )


        train_indices = permutation[
            :n_train
        ]


        val_indices = permutation[
            n_train:
        ]


        train_group = group.iloc[
            train_indices
        ].copy()


        val_group = group.iloc[
            val_indices
        ].copy()


        train_group[
            "split"
        ] = "train"


        val_group[
            "split"
        ] = "validation"


        train_parts.append(
            train_group
        )


        val_parts.append(
            val_group
        )


    train_split = pd.concat(
        train_parts,
        ignore_index=True
    )


    val_split = pd.concat(
        val_parts,
        ignore_index=True
    )


    return (
        train_split,
        val_split
    )


# ==================================================================================================
# 17. BUILD ALL CANDIDATE DATASETS
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "8. BUILD STRATIFIED 70/30 CANDIDATE DATASETS"
)

print(
    "=" * 150
)


threshold_rows = []


for threshold in SUPPORT_THRESHOLDS:

    print()

    print(
        "-" * 150
    )

    print(
        f"SUPPORT >= {threshold}"
    )

    print(
        "-" * 150
    )


    eligible_glosses = set(

        support.loc[

            support[
                "support"
            ]
            >=
            threshold,

            "gloss"

        ]

    )


    candidate_df = clean_df[
        clean_df[
            "gloss"
        ].isin(
            eligible_glosses
        )
    ].copy()


    candidate_df = (
        candidate_df
        .sort_values(
            [
                "gloss",
                "uid"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    n_classes = candidate_df[
        "gloss"
    ].nunique()


    n_videos = len(
        candidate_df
    )


    # ----------------------------------------------------------------------------------------------
    # Create deterministic split
    # ----------------------------------------------------------------------------------------------

    train_split, val_split = (
        create_stratified_split(

            candidate_df,

            seed=SEED,

            train_fraction=
                TRAIN_FRACTION

        )
    )


    # ----------------------------------------------------------------------------------------------
    # Integrity
    # ----------------------------------------------------------------------------------------------

    train_uids = set(
        train_split[
            "uid"
        ]
    )


    val_uids = set(
        val_split[
            "uid"
        ]
    )


    overlap = (
        train_uids
        &
        val_uids
    )


    train_classes = set(
        train_split[
            "gloss"
        ]
    )


    val_classes = set(
        val_split[
            "gloss"
        ]
    )


    class_difference = (
        train_classes
        ^
        val_classes
    )


    if overlap:

        raise RuntimeError(

            f"UID leakage for "
            f"support >= {threshold}"

        )


    if class_difference:

        raise RuntimeError(

            f"Class mismatch for "
            f"support >= {threshold}"

        )


    if len(
        train_classes
    ) != n_classes:

        raise RuntimeError(
            "Train class count mismatch."
        )


    if len(
        val_classes
    ) != n_classes:

        raise RuntimeError(
            "Validation class count mismatch."
        )


    # ----------------------------------------------------------------------------------------------
    # Class IDs
    # ----------------------------------------------------------------------------------------------

    ordered_glosses = sorted(
        train_classes
    )


    class_mapping = {

        gloss:
            class_id

        for class_id, gloss in enumerate(
            ordered_glosses
        )

    }


    train_split[
        "class_id"
    ] = (
        train_split[
            "gloss"
        ]
        .map(
            class_mapping
        )
        .astype(
            int
        )
    )


    val_split[
        "class_id"
    ] = (
        val_split[
            "gloss"
        ]
        .map(
            class_mapping
        )
        .astype(
            int
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Paths
    # ----------------------------------------------------------------------------------------------

    candidate_root = (
        SPLIT_DIR /
        f"support_ge_{threshold}"
    )


    candidate_root.mkdir(
        parents=True,
        exist_ok=True
    )


    train_path = (
        candidate_root /
        "train_70.csv"
    )


    val_path = (
        candidate_root /
        "validation_30.csv"
    )


    mapping_path = (
        candidate_root /
        "class_mapping.json"
    )


    train_split.to_csv(
        train_path,
        index=False
    )


    val_split.to_csv(
        val_path,
        index=False
    )


    # ----------------------------------------------------------------------------------------------
    # Mapping JSON
    # ----------------------------------------------------------------------------------------------

    mapping_json = {

        "support_threshold":
            threshold,

        "num_classes":
            int(
                n_classes
            ),

        "gloss_to_class_id":
            class_mapping,

        "class_id_to_gloss": {

            str(
                class_id
            ):
                gloss

            for gloss, class_id
            in class_mapping.items()

        }

    }


    save_json(
        mapping_json,
        mapping_path
    )


    # ----------------------------------------------------------------------------------------------
    # Statistics
    # ----------------------------------------------------------------------------------------------

    actual_train_fraction = (

        len(
            train_split
        )
        /
        n_videos

    )


    actual_val_fraction = (

        len(
            val_split
        )
        /
        n_videos

    )


    minimum_train_per_class = int(

        train_split
        .groupby(
            "gloss"
        )
        .size()
        .min()

    )


    minimum_val_per_class = int(

        val_split
        .groupby(
            "gloss"
        )
        .size()
        .min()

    )


    maximum_train_per_class = int(

        train_split
        .groupby(
            "gloss"
        )
        .size()
        .max()

    )


    maximum_val_per_class = int(

        val_split
        .groupby(
            "gloss"
        )
        .size()
        .max()

    )


    threshold_rows.append(
        {

            "minimum_support":
                threshold,

            "classes":
                int(
                    n_classes
                ),

            "total_videos":
                int(
                    n_videos
                ),

            "train_videos":
                int(
                    len(
                        train_split
                    )
                ),

            "validation_videos":
                int(
                    len(
                        val_split
                    )
                ),

            "actual_train_percent":
                float(
                    actual_train_fraction *
                    100.0
                ),

            "actual_validation_percent":
                float(
                    actual_val_fraction *
                    100.0
                ),

            "min_train_per_class":
                minimum_train_per_class,

            "max_train_per_class":
                maximum_train_per_class,

            "min_validation_per_class":
                minimum_val_per_class,

            "max_validation_per_class":
                maximum_val_per_class,

            "uid_overlap":
                len(
                    overlap
                ),

            "train_classes":
                len(
                    train_classes
                ),

            "validation_classes":
                len(
                    val_classes
                ),

            "train_manifest":
                str(
                    train_path
                ),

            "validation_manifest":
                str(
                    val_path
                ),

            "class_mapping":
                str(
                    mapping_path
                )

        }
    )


    print(
        "Classes:",
        n_classes
    )


    print(
        "Total videos:",
        n_videos
    )


    print(
        "Train:",
        len(
            train_split
        ),
        f"({actual_train_fraction * 100:.2f}%)"
    )


    print(
        "Validation:",
        len(
            val_split
        ),
        f"({actual_val_fraction * 100:.2f}%)"
    )


    print(
        "Min train/class:",
        minimum_train_per_class
    )


    print(
        "Min validation/class:",
        minimum_val_per_class
    )


    print(
        "UID overlap:",
        len(
            overlap
        )
    )


    print(
        "[PASS] Candidate split valid."
    )


# ==================================================================================================
# 18. THRESHOLD SUMMARY
# ==================================================================================================

threshold_summary = pd.DataFrame(
    threshold_rows
)


threshold_summary.to_csv(
    THRESHOLD_SUMMARY_PATH,
    index=False
)


print(
    "\n" +
    "=" * 150
)

print(
    "9. FULL SUPERVISED CANDIDATE SUMMARY"
)

print(
    "=" * 150
)


display_columns = [

    "minimum_support",
    "classes",
    "total_videos",
    "train_videos",
    "validation_videos",
    "actual_train_percent",
    "actual_validation_percent",
    "min_train_per_class",
    "min_validation_per_class"

]


print(
    threshold_summary[
        display_columns
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 19. DATASET COVERAGE
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "10. FULL-DATASET COVERAGE"
)

print(
    "=" * 150
)


coverage_rows = []


for _, row in (
    threshold_summary.iterrows()
):

    coverage = (

        int(
            row[
                "total_videos"
            ]
        )
        /
        max(
            len(
                clean_df
            ),
            1
        )

        *
        100.0

    )


    coverage_rows.append(
        coverage
    )


threshold_summary[
    "dataset_video_coverage_percent"
] = coverage_rows


threshold_summary.to_csv(
    THRESHOLD_SUMMARY_PATH,
    index=False
)


for _, row in (
    threshold_summary.iterrows()
):

    print(

        f"support >= "
        f"{int(row['minimum_support']):2d}: "

        f"{int(row['total_videos']):5d}/"
        f"{len(clean_df):5d} videos | "

        f"{row['dataset_video_coverage_percent']:.2f}% coverage"

    )


# ==================================================================================================
# 20. RARE-CLASS POPULATION
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "11. RARE-CLASS POPULATION"
)

print(
    "=" * 150
)


singleton_classes = int(
    (
        support[
            "support"
        ]
        ==
        1
    ).sum()
)


singleton_videos = int(

    support.loc[

        support[
            "support"
        ]
        ==
        1,

        "support"

    ].sum()

)


print(
    "Singleton classes:",
    singleton_classes
)


print(
    "Singleton videos:",
    singleton_videos
)


print()


print(
    "[NOTE]"
)

print(
    "Singleton classes cannot participate in an ordinary"
)

print(
    "supervised train/validation classifier because the same class"
)

print(
    "cannot independently appear in both partitions."
)


# ==================================================================================================
# 21. SUMMARY
# ==================================================================================================

summary = {

    "stage":
        "L20A",

    "status":
        "COMPLETE",

    "task":
        (
            "full_cislr_supervised_dataset_audit"
        ),

    "seed":
        SEED,

    "split_protocol": {

        "train_fraction_target":
            TRAIN_FRACTION,

        "validation_fraction_target":
            VALIDATION_FRACTION,

        "stratified_by":
            "gloss",

        "minimum_train_per_class":
            1,

        "minimum_validation_per_class":
            1

    },

    "full_dataset": {

        "metadata_rows":
            int(
                len(
                    df
                )
            ),

        "metadata_valid":
            int(
                len(
                    valid_df
                )
            ),

        "video_backed_clean_samples":
            int(
                len(
                    clean_df
                )
            ),

        "unique_classes":
            int(
                clean_df[
                    "gloss"
                ].nunique()
            ),

        "singleton_classes":
            singleton_classes,

        "singleton_videos":
            singleton_videos

    },

    "candidate_thresholds":
        [

            {

                "minimum_support":
                    int(
                        row[
                            "minimum_support"
                        ]
                    ),

                "classes":
                    int(
                        row[
                            "classes"
                        ]
                    ),

                "total_videos":
                    int(
                        row[
                            "total_videos"
                        ]
                    ),

                "train_videos":
                    int(
                        row[
                            "train_videos"
                        ]
                    ),

                "validation_videos":
                    int(
                        row[
                            "validation_videos"
                        ]
                    ),

                "video_coverage_percent":
                    float(
                        row[
                            "dataset_video_coverage_percent"
                        ]
                    )

            }

            for _, row
            in threshold_summary.iterrows()

        ],

    "model_training_performed":
        False,

    "model_inference_performed":
        False,

    "accuracy_used_for_dataset_selection":
        False,

    "next_stage":
        (
            "L20B freeze supervised dataset design"
        ),

    "created_at":
        datetime.now().isoformat()

}


save_json(
    summary,
    SUMMARY_PATH
)


# ==================================================================================================
# 22. HASH OUTPUTS
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "12. ARTIFACT HASHING"
)

print(
    "=" * 150
)


important_artifacts = {

    "clean_manifest":
        CLEAN_MANIFEST_PATH,

    "invalid_manifest":
        INVALID_MANIFEST_PATH,

    "support_table":
        SUPPORT_TABLE_PATH,

    "threshold_summary":
        THRESHOLD_SUMMARY_PATH,

    "summary":
        SUMMARY_PATH

}


hash_manifest = {}


for name, path in (
    important_artifacts.items()
):

    digest = sha256_file(
        path
    )


    hash_manifest[
        name
    ] = {

        "path":
            str(
                path
            ),

        "sha256":
            digest

    }


    print(

        f"{name:25s}: "
        f"{digest}"

    )


save_json(
    hash_manifest,
    HASH_MANIFEST_PATH
)


# ==================================================================================================
# 23. MASTER FREEZE
# ==================================================================================================

master_freeze = {

    "stage":
        "L20A",

    "status":
        "COMPLETE_AND_FROZEN",

    "purpose":
        (
            "Full CISLR supervised dataset "
            "feasibility audit"
        ),

    "seed":
        SEED,

    "metadata_source":
        str(
            DATASET_CSV
        ),

    "video_source":
        str(
            VIDEO_ROOT
        ),

    "clean_population": {

        "videos":
            int(
                len(
                    clean_df
                )
            ),

        "classes":
            int(
                clean_df[
                    "gloss"
                ].nunique()
            )

    },

    "split_policy":
        "DETERMINISTIC_STRATIFIED_70_30",

    "candidate_thresholds":
        SUPPORT_THRESHOLDS,

    "model_training":
        False,

    "model_inference":
        False,

    "accuracy_used":
        False,

    "next_stage":
        "L20B_DATASET_DESIGN_FREEZE",

    "created_at":
        datetime.now().isoformat()

}


save_json(
    master_freeze,
    FREEZE_PATH
)


MASTER_SHA = sha256_file(
    FREEZE_PATH
)


# ==================================================================================================
# 24. FINAL INTEGRITY CHECK
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "13. FINAL L20A INTEGRITY CHECK"
)

print(
    "=" * 150
)


checks = {

    "dataset_loaded":
        len(
            df
        ) > 0,

    "clean_dataset_nonempty":
        len(
            clean_df
        ) > 0,

    "classes_found":
        clean_df[
            "gloss"
        ].nunique()
        >
        0,

    "support_table_saved":
        SUPPORT_TABLE_PATH.exists(),

    "threshold_summary_saved":
        THRESHOLD_SUMMARY_PATH.exists(),

    "candidate_splits_created":
        len(
            threshold_summary
        )
        ==
        len(
            SUPPORT_THRESHOLDS
        ),

    "no_model_training":
        True,

    "no_model_inference":
        True,

    "accuracy_not_used":
        True,

    "freeze_saved":
        FREEZE_PATH.exists()

}


for name, passed in (
    checks.items()
):

    print(

        f"{name:35s}: "
        f"{'PASS' if passed else 'FAIL'}"

    )


failed = [

    name

    for name, passed
    in checks.items()

    if not passed

]


if failed:

    raise RuntimeError(

        f"L20A integrity failure: "
        f"{failed}"

    )


# ==================================================================================================
# 25. FINAL REPORT
# ==================================================================================================

print(
    "\n"
)

print(
    "=" * 150
)

print(
    "L20A — FULL CISLR SUPERVISED DATASET AUDIT COMPLETE"
)

print(
    "=" * 150
)

print()


print(
    "FULL DATASET"
)


print(
    f"  Metadata rows             : {len(df)}"
)


print(
    f"  Valid metadata rows       : {len(valid_df)}"
)


print(
    f"  Clean video-backed rows   : {len(clean_df)}"
)


print(
    f"  Unique valid classes      : {clean_df['gloss'].nunique()}"
)


print(
    f"  Singleton classes         : {singleton_classes}"
)


print()


print(
    "70/30 SUPERVISED CANDIDATES"
)

print(
    "-" * 120
)


print(

    threshold_summary[

        [
            "minimum_support",
            "classes",
            "total_videos",
            "train_videos",
            "validation_videos",
            "dataset_video_coverage_percent"
        ]

    ].to_string(
        index=False
    )

)


print(
    "-" * 120
)

print()


print(
    "OUTPUT"
)


print(
    "  Support report:"
)

print(
    f"    {SUPPORT_TABLE_PATH}"
)


print()


print(
    "  Threshold comparison:"
)

print(
    f"    {THRESHOLD_SUMMARY_PATH}"
)


print()


print(
    "  Candidate split root:"
)

print(
    f"    {SPLIT_DIR}"
)


print()


print(
    "  Master freeze:"
)

print(
    f"    {FREEZE_PATH}"
)


print()


print(
    "  MASTER SHA256:"
)

print(
    f"    {MASTER_SHA}"
)


print()


print(
    "[PASS] L20A COMPLETE AND FROZEN"
)


print(
    "[PASS] Full CISLR metadata audited"
)


print(
    "[PASS] Video-backed population established"
)


print(
    "[PASS] Stratified 70/30 candidate datasets generated"
)


print(
    "[PASS] Every retained class exists in TRAIN and VALIDATION"
)


print(
    "[PASS] Zero train/validation UID overlap"
)


print(
    "[PASS] No model training performed"
)


print(
    "[PASS] No accuracy used to choose dataset"
)


print()


print(
    "[NEXT] L20B = FREEZE FULL-CISLR SUPERVISED DATASET DESIGN"
)

print(
    "=" * 150
)

CISLR — L20A
FULL CISLR SUPERVISED DATASET AUDIT + STRATIFIED 70/30 CANDIDATE SPLITS

Python: 3.11.15
Pandas: 3.0.5
OpenCV: 5.0.0
Seed: 42
Train fraction: 0.7
Validation fraction: 0.3

1. REQUIRED INPUT CHECK
dataset.csv                   : PASS
CISLR video directory         : PASS

[PASS] Required CISLR inputs found.

2. LOAD FULL CISLR METADATA
Metadata rows: 7050
Columns: ['uid', 'gloss', 'duration', 'category']

3. METADATA VALIDITY
Metadata total: 7050
Metadata valid: 7048
Metadata invalid: 2

4. UID INTEGRITY
Duplicate UID rows: 0
Unique valid UIDs: 7048

5. BUILD CISLR VIDEO INDEX
MP4 files discovered: 7050
Unique MP4 stems: 7050
Duplicate MP4 stems: 0

6. RESOLVE METADATA → VIDEO
Videos resolved: 7048
Videos missing: 0

Clean video-backed samples: 7048
Clean unique glosses: 4763

7. FULL CLASS-SUPPORT DISTRIBUTION
Unique classes: 4763

support >=  1:  4763 classes |  7048 videos
support >=  2:  1363 classes |  3648 videos
support >=  3:   543 classes |  2008 videos
support >=  

In [2]:
# ==================================================================================================
# CISLR — L20B
#
# FREEZE FULL-CISLR LEARNING PROTOCOL
# + CORRECTED SUPPORT>=4 SUPERVISED DATASET
# + GLOBAL NEAR-70/30 STRATIFIED SPLIT
# + FULL 7048-VIDEO DOMAIN-PRETRAINING MANIFEST
#
# --------------------------------------------------------------------------------------------------
# INPUT
# --------------------------------------------------------------------------------------------------
#
# L20A established:
#
#   clean video-backed CISLR samples : 7048
#   valid glosses                    : 4763
#   singleton classes                : 3400
#
# Selected supervised criterion:
#
#   minimum support >= 4
#
# Expected supervised population:
#
#   215 classes
#   1024 independent videos
#
# --------------------------------------------------------------------------------------------------
# L20B PURPOSE
# --------------------------------------------------------------------------------------------------
#
# 1. Verify L20A artifacts.
# 2. Freeze support >= 4 BEFORE model training.
# 3. Create full 7048-video domain-pretraining manifest.
# 4. Create 215-class supervised population.
# 5. Construct deterministic near-70/30 train/validation split.
# 6. Guarantee every supervised class exists in BOTH train and validation.
# 7. Guarantee zero UID overlap.
# 8. Freeze class mapping.
# 9. Hash all critical artifacts.
#
# --------------------------------------------------------------------------------------------------
# IMPORTANT
# --------------------------------------------------------------------------------------------------
#
# NO MODEL TRAINING
# NO MODEL INFERENCE
# NO ACCURACY CHECKING
# NO TEST-BASED SELECTION
# NO SEED SEARCHING
#
# Dataset design is frozen before seeing L20C/L20D model performance.
#
# ==================================================================================================


# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

import os
import json
import math
import random
import hashlib
import platform

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# ==================================================================================================
# 2. FROZEN L20B DESIGN
# ==================================================================================================

SUPPORT_THRESHOLD = 4

TARGET_TRAIN_FRACTION = 0.70
TARGET_VALIDATION_FRACTION = 0.30

EXPECTED_SUPERVISED_CLASSES = 215
EXPECTED_SUPERVISED_VIDEOS = 1024

EXPECTED_FULL_CLEAN_VIDEOS = 7048
EXPECTED_FULL_CLASSES = 4763


# ==================================================================================================
# 3. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)


L20A_ROOT = (
    ROOT /
    "audit/l20a_full_cislr_dataset_audit"
)


L20A_CLEAN_MANIFEST = (
    L20A_ROOT /
    "manifests/l20a_clean_full_cislr_manifest.csv"
)


L20A_SUPPORT_TABLE = (
    L20A_ROOT /
    "reports/l20a_class_support_table.csv"
)


L20A_THRESHOLD_SUMMARY = (
    L20A_ROOT /
    "reports/l20a_support_threshold_summary.csv"
)


L20A_FREEZE = (
    L20A_ROOT /
    "L20A_MASTER_FREEZE.json"
)


# --------------------------------------------------------------------------------------------------
# L20B outputs
# --------------------------------------------------------------------------------------------------

L20B_ROOT = (
    ROOT /
    "audit/l20b_full_cislr_protocol_freeze"
)


MANIFEST_DIR = (
    L20B_ROOT /
    "manifests"
)


CONFIG_DIR = (
    L20B_ROOT /
    "config"
)


REPORT_DIR = (
    L20B_ROOT /
    "reports"
)


HASH_DIR = (
    L20B_ROOT /
    "hashes"
)


for directory in [

    L20B_ROOT,
    MANIFEST_DIR,
    CONFIG_DIR,
    REPORT_DIR,
    HASH_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 4. OUTPUT FILES
# ==================================================================================================

FULL_PRETRAIN_MANIFEST_PATH = (
    MANIFEST_DIR /
    "l20b_full_7048_domain_pretraining_manifest.csv"
)


SUPERVISED_MASTER_PATH = (
    MANIFEST_DIR /
    "l20b_support_ge4_supervised_master.csv"
)


TRAIN_MANIFEST_PATH = (
    MANIFEST_DIR /
    "l20b_train_manifest.csv"
)


VALIDATION_MANIFEST_PATH = (
    MANIFEST_DIR /
    "l20b_validation_manifest.csv"
)


CLASS_MAPPING_PATH = (
    CONFIG_DIR /
    "l20b_215_class_mapping.json"
)


SPLIT_CONFIG_PATH = (
    CONFIG_DIR /
    "l20b_split_config.json"
)


PRETRAIN_CONFIG_PATH = (
    CONFIG_DIR /
    "l20c_pretraining_config.json"
)


SUPERVISED_CONFIG_PATH = (
    CONFIG_DIR /
    "l20d_supervised_training_config.json"
)


PER_CLASS_SPLIT_PATH = (
    REPORT_DIR /
    "l20b_per_class_split_counts.csv"
)


SUMMARY_PATH = (
    REPORT_DIR /
    "l20b_summary.json"
)


HASH_MANIFEST_PATH = (
    HASH_DIR /
    "l20b_sha256_manifest.json"
)


FREEZE_PATH = (
    L20B_ROOT /
    "L20B_MASTER_FREEZE.json"
)


# ==================================================================================================
# 5. HELPERS
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:

                break

            h.update(
                block
            )

    return h.hexdigest()


def save_json(
    obj,
    path
):

    with open(
        path,
        "w"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )


def assert_unique_uid(
    frame,
    name
):

    duplicate_count = int(
        frame[
            "uid"
        ].duplicated().sum()
    )

    if duplicate_count != 0:

        raise RuntimeError(
            f"{name}: duplicate UID count = "
            f"{duplicate_count}"
        )


# ==================================================================================================
# 6. HEADER
# ==================================================================================================

print(
    "=" * 150
)

print(
    "CISLR — L20B"
)

print(
    "FULL-CISLR LEARNING PROTOCOL FREEZE"
)

print(
    "SUPPORT >= 4 | 215-CLASS SUPERVISED DATASET | CORRECTED NEAR-70/30 SPLIT"
)

print(
    "=" * 150
)

print()

print(
    "Python:",
    platform.python_version()
)

print(
    "Pandas:",
    pd.__version__
)

print(
    "NumPy:",
    np.__version__
)

print(
    "Seed:",
    SEED
)

print(
    "Frozen support threshold:",
    SUPPORT_THRESHOLD
)

print(
    "Target train fraction:",
    TARGET_TRAIN_FRACTION
)

print(
    "Target validation fraction:",
    TARGET_VALIDATION_FRACTION
)

print()


# ==================================================================================================
# 7. VERIFY L20A ARTIFACTS
# ==================================================================================================

print(
    "=" * 150
)

print(
    "1. VERIFY L20A INPUT ARTIFACTS"
)

print(
    "=" * 150
)


required_inputs = {

    "L20A clean manifest":
        L20A_CLEAN_MANIFEST,

    "L20A support table":
        L20A_SUPPORT_TABLE,

    "L20A threshold summary":
        L20A_THRESHOLD_SUMMARY,

    "L20A master freeze":
        L20A_FREEZE

}


for name, path in (
    required_inputs.items()
):

    exists = path.exists()

    print(
        f"{name:35s}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    if not exists:

        raise FileNotFoundError(
            path
        )


# ==================================================================================================
# 8. VERIFY L20A FREEZE
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "2. VERIFY L20A FREEZE"
)

print(
    "=" * 150
)


with open(
    L20A_FREEZE,
    "r"
) as f:

    l20a_freeze = json.load(
        f
    )


print(
    "L20A stage:",
    l20a_freeze.get(
        "stage"
    )
)


print(
    "L20A status:",
    l20a_freeze.get(
        "status"
    )
)


if (
    l20a_freeze.get(
        "stage"
    )
    !=
    "L20A"
):

    raise RuntimeError(
        "Invalid L20A freeze stage."
    )


if (
    l20a_freeze.get(
        "status"
    )
    !=
    "COMPLETE_AND_FROZEN"
):

    raise RuntimeError(
        "L20A is not COMPLETE_AND_FROZEN."
    )


print(
    "[PASS] L20A freeze verified."
)


# ==================================================================================================
# 9. LOAD L20A CLEAN POPULATION
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "3. LOAD FULL CLEAN CISLR POPULATION"
)

print(
    "=" * 150
)


full_df = pd.read_csv(
    L20A_CLEAN_MANIFEST
)


full_df[
    "uid"
] = (
    full_df[
        "uid"
    ]
    .astype(str)
    .str.strip()
)


full_df[
    "gloss"
] = (
    full_df[
        "gloss"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


assert_unique_uid(
    full_df,
    "full CISLR population"
)


full_video_count = len(
    full_df
)


full_class_count = full_df[
    "gloss"
].nunique()


print(
    "Clean videos:",
    full_video_count
)


print(
    "Unique classes:",
    full_class_count
)


if (
    full_video_count
    !=
    EXPECTED_FULL_CLEAN_VIDEOS
):

    raise RuntimeError(

        f"Expected "
        f"{EXPECTED_FULL_CLEAN_VIDEOS} "
        f"clean videos, found "
        f"{full_video_count}."

    )


if (
    full_class_count
    !=
    EXPECTED_FULL_CLASSES
):

    raise RuntimeError(

        f"Expected "
        f"{EXPECTED_FULL_CLASSES} "
        f"classes, found "
        f"{full_class_count}."

    )


print(
    "[PASS] Full L20A population matches frozen audit."
)


# ==================================================================================================
# 10. CREATE FULL DOMAIN-PRETRAINING MANIFEST
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "4. CREATE FULL 7048-VIDEO DOMAIN-PRETRAINING MANIFEST"
)

print(
    "=" * 150
)


pretrain_df = (
    full_df
    .copy()
    .sort_values(
        [
            "gloss",
            "uid"
        ]
    )
    .reset_index(
        drop=True
    )
)


pretrain_df[
    "domain_pretraining_eligible"
] = True


pretrain_df[
    "domain_pretraining_role"
] = (
    "unlabeled_or_label_agnostic_pretraining"
)


pretrain_df.to_csv(
    FULL_PRETRAIN_MANIFEST_PATH,
    index=False
)


print(
    "Domain-pretraining videos:",
    len(
        pretrain_df
    )
)


print(
    "Domain-pretraining gloss population:",
    pretrain_df[
        "gloss"
    ].nunique()
)


print(
    "[PASS] All 7048 clean CISLR videos assigned to domain-pretraining population."
)


# ==================================================================================================
# 11. RECOMPUTE SUPPORT FROM CLEAN POPULATION
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "5. RECOMPUTE CLASS SUPPORT"
)

print(
    "=" * 150
)


support = (
    full_df
    .groupby(
        "gloss"
    )
    .size()
    .reset_index(
        name="support"
    )
)


eligible_support = support[
    support[
        "support"
    ]
    >=
    SUPPORT_THRESHOLD
].copy()


eligible_glosses = set(
    eligible_support[
        "gloss"
    ]
)


print(
    "Support >= 4 classes:",
    len(
        eligible_glosses
    )
)


print(
    "Minimum retained support:",
    eligible_support[
        "support"
    ].min()
)


print(
    "Maximum retained support:",
    eligible_support[
        "support"
    ].max()
)


# ==================================================================================================
# 12. BUILD SUPERVISED MASTER POPULATION
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "6. BUILD FROZEN SUPERVISED POPULATION"
)

print(
    "=" * 150
)


supervised_df = full_df[
    full_df[
        "gloss"
    ].isin(
        eligible_glosses
    )
].copy()


supervised_df = (
    supervised_df
    .sort_values(
        [
            "gloss",
            "uid"
        ]
    )
    .reset_index(
        drop=True
    )
)


supervised_class_count = supervised_df[
    "gloss"
].nunique()


supervised_video_count = len(
    supervised_df
)


print(
    "Supervised classes:",
    supervised_class_count
)


print(
    "Supervised videos:",
    supervised_video_count
)


if (
    supervised_class_count
    !=
    EXPECTED_SUPERVISED_CLASSES
):

    raise RuntimeError(

        f"Expected "
        f"{EXPECTED_SUPERVISED_CLASSES} "
        f"supervised classes, found "
        f"{supervised_class_count}."

    )


if (
    supervised_video_count
    !=
    EXPECTED_SUPERVISED_VIDEOS
):

    raise RuntimeError(

        f"Expected "
        f"{EXPECTED_SUPERVISED_VIDEOS} "
        f"supervised videos, found "
        f"{supervised_video_count}."

    )


assert_unique_uid(
    supervised_df,
    "supervised population"
)


print(
    "[PASS] Frozen supervised population = 215 classes / 1024 videos."
)


# ==================================================================================================
# 13. FREEZE CLASS MAPPING
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "7. FREEZE 215-CLASS MAPPING"
)

print(
    "=" * 150
)


ordered_glosses = sorted(
    eligible_glosses
)


gloss_to_class_id = {

    gloss:
        class_id

    for class_id, gloss
    in enumerate(
        ordered_glosses
    )

}


class_id_to_gloss = {

    str(
        class_id
    ):
        gloss

    for gloss, class_id
    in gloss_to_class_id.items()

}


supervised_df[
    "class_id"
] = (
    supervised_df[
        "gloss"
    ]
    .map(
        gloss_to_class_id
    )
    .astype(
        int
    )
)


class_mapping = {

    "stage":
        "L20B",

    "status":
        "FROZEN",

    "support_threshold":
        SUPPORT_THRESHOLD,

    "num_classes":
        len(
            ordered_glosses
        ),

    "gloss_to_class_id":
        gloss_to_class_id,

    "class_id_to_gloss":
        class_id_to_gloss

}


save_json(
    class_mapping,
    CLASS_MAPPING_PATH
)


print(
    "Classes mapped:",
    len(
        ordered_glosses
    )
)


print(
    "Class IDs:",
    f"0 .. {len(ordered_glosses)-1}"
)


print(
    "[PASS] Class mapping frozen."
)


# ==================================================================================================
# 14. DETERMINE TARGET GLOBAL TRAIN COUNT
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "8. DETERMINE CORRECTED GLOBAL 70/30 TARGET"
)

print(
    "=" * 150
)


N = len(
    supervised_df
)


TARGET_TRAIN_COUNT = int(
    round(
        N *
        TARGET_TRAIN_FRACTION
    )
)


TARGET_VALIDATION_COUNT = (
    N
    -
    TARGET_TRAIN_COUNT
)


print(
    "Total supervised videos:",
    N
)


print(
    "Target train videos:",
    TARGET_TRAIN_COUNT
)


print(
    "Target validation videos:",
    TARGET_VALIDATION_COUNT
)


print(
    "Target train %:",
    (
        TARGET_TRAIN_COUNT
        /
        N
        *
        100.0
    )
)


print(
    "Target validation %:",
    (
        TARGET_VALIDATION_COUNT
        /
        N
        *
        100.0
    )
)


# ==================================================================================================
# 15. INITIAL PER-CLASS INTEGER ALLOCATION
# ==================================================================================================
#
# For each class:
#
#     ideal_train = support * 0.70
#
# Start with floor(ideal_train), constrained to:
#
#     >=1 train
#     >=1 validation
#
# Then globally adjust allocations until total train count reaches
# TARGET_TRAIN_COUNT.
#
# Classes whose fractional ideal_train component is largest receive
# the additional training sample first.
#
# This is deterministic.
#
# ==================================================================================================

class_allocations = []


for gloss in ordered_glosses:

    n = int(
        (
            supervised_df[
                "gloss"
            ]
            ==
            gloss
        ).sum()
    )


    ideal_train = (
        n *
        TARGET_TRAIN_FRACTION
    )


    floor_train = int(
        math.floor(
            ideal_train
        )
    )


    floor_train = max(
        1,
        floor_train
    )


    floor_train = min(
        n - 1,
        floor_train
    )


    class_allocations.append(
        {

            "gloss":
                gloss,

            "class_id":
                int(
                    gloss_to_class_id[
                        gloss
                    ]
                ),

            "support":
                n,

            "ideal_train":
                float(
                    ideal_train
                ),

            "fractional_remainder":
                float(
                    ideal_train
                    -
                    math.floor(
                        ideal_train
                    )
                ),

            "train_count":
                int(
                    floor_train
                )

        }
    )


allocation_df = pd.DataFrame(
    class_allocations
)


initial_train_total = int(
    allocation_df[
        "train_count"
    ].sum()
)


print()

print(
    "Initial constrained train total:",
    initial_train_total
)


print(
    "Desired train total:",
    TARGET_TRAIN_COUNT
)


# ==================================================================================================
# 16. CORRECT GLOBAL ALLOCATION
# ==================================================================================================

difference = (
    TARGET_TRAIN_COUNT
    -
    initial_train_total
)


print(
    "Required allocation adjustment:",
    difference
)


if difference > 0:

    # ----------------------------------------------------------------------------------------------
    # Add training examples to classes closest to needing an extra example
    # according to their fractional 70% allocation.
    # ----------------------------------------------------------------------------------------------

    candidates = (
        allocation_df[
            allocation_df[
                "train_count"
            ]
            <
            (
                allocation_df[
                    "support"
                ]
                -
                1
            )
        ]
        .sort_values(
            [
                "fractional_remainder",
                "support",
                "gloss"
            ],
            ascending=[
                False,
                False,
                True
            ]
        )
        .index
        .tolist()
    )


    if len(
        candidates
    ) < difference:

        raise RuntimeError(
            "Not enough classes available to reach target train count."
        )


    for idx in candidates[
        :difference
    ]:

        allocation_df.loc[
            idx,
            "train_count"
        ] += 1


elif difference < 0:

    # ----------------------------------------------------------------------------------------------
    # Remove training examples from classes whose ideal allocation is
    # least harmed by moving one example to validation.
    # ----------------------------------------------------------------------------------------------

    remove_count = abs(
        difference
    )


    candidates = (
        allocation_df[
            allocation_df[
                "train_count"
            ]
            >
            1
        ]
        .sort_values(
            [
                "fractional_remainder",
                "support",
                "gloss"
            ],
            ascending=[
                True,
                True,
                True
            ]
        )
        .index
        .tolist()
    )


    if len(
        candidates
    ) < remove_count:

        raise RuntimeError(
            "Not enough classes available to reduce train count."
        )


    for idx in candidates[
        :remove_count
    ]:

        allocation_df.loc[
            idx,
            "train_count"
        ] -= 1


allocation_df[
    "validation_count"
] = (
    allocation_df[
        "support"
    ]
    -
    allocation_df[
        "train_count"
    ]
)


final_allocated_train = int(
    allocation_df[
        "train_count"
    ].sum()
)


final_allocated_validation = int(
    allocation_df[
        "validation_count"
    ].sum()
)


print()

print(
    "Final allocated train:",
    final_allocated_train
)


print(
    "Final allocated validation:",
    final_allocated_validation
)


if (
    final_allocated_train
    !=
    TARGET_TRAIN_COUNT
):

    raise RuntimeError(
        "Failed to reach target train count."
    )


if (
    final_allocated_validation
    !=
    TARGET_VALIDATION_COUNT
):

    raise RuntimeError(
        "Failed to reach target validation count."
    )


if (
    allocation_df[
        "train_count"
    ].min()
    <
    1
):

    raise RuntimeError(
        "A class has zero training samples."
    )


if (
    allocation_df[
        "validation_count"
    ].min()
    <
    1
):

    raise RuntimeError(
        "A class has zero validation samples."
    )


print(
    "[PASS] Global near-70/30 allocation solved."
)


# ==================================================================================================
# 17. DETERMINISTIC WITHIN-CLASS SPLIT
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "9. CREATE DETERMINISTIC STRATIFIED SPLIT"
)

print(
    "=" * 150
)


rng = np.random.default_rng(
    SEED
)


train_parts = []

validation_parts = []


for _, allocation in (
    allocation_df
    .sort_values(
        "class_id"
    )
    .iterrows()
):

    gloss = allocation[
        "gloss"
    ]


    n_train = int(
        allocation[
            "train_count"
        ]
    )


    class_df = (
        supervised_df[
            supervised_df[
                "gloss"
            ]
            ==
            gloss
        ]
        .copy()
        .sort_values(
            "uid"
        )
        .reset_index(
            drop=True
        )
    )


    permutation = rng.permutation(
        len(
            class_df
        )
    )


    train_indices = permutation[
        :n_train
    ]


    validation_indices = permutation[
        n_train:
    ]


    class_train = class_df.iloc[
        train_indices
    ].copy()


    class_validation = class_df.iloc[
        validation_indices
    ].copy()


    class_train[
        "split"
    ] = "train"


    class_validation[
        "split"
    ] = "validation"


    train_parts.append(
        class_train
    )


    validation_parts.append(
        class_validation
    )


train_df = pd.concat(
    train_parts,
    ignore_index=True
)


validation_df = pd.concat(
    validation_parts,
    ignore_index=True
)


# Sort output only AFTER membership has been determined.

train_df = (
    train_df
    .sort_values(
        [
            "class_id",
            "uid"
        ]
    )
    .reset_index(
        drop=True
    )
)


validation_df = (
    validation_df
    .sort_values(
        [
            "class_id",
            "uid"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# 18. SPLIT INTEGRITY
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "10. SPLIT INTEGRITY"
)

print(
    "=" * 150
)


assert_unique_uid(
    train_df,
    "train"
)


assert_unique_uid(
    validation_df,
    "validation"
)


train_uids = set(
    train_df[
        "uid"
    ]
)


validation_uids = set(
    validation_df[
        "uid"
    ]
)


uid_overlap = (
    train_uids
    &
    validation_uids
)


train_classes = set(
    train_df[
        "gloss"
    ]
)


validation_classes = set(
    validation_df[
        "gloss"
    ]
)


master_classes = set(
    supervised_df[
        "gloss"
    ]
)


train_validation_union = (
    train_uids
    |
    validation_uids
)


master_uids = set(
    supervised_df[
        "uid"
    ]
)


checks = {

    "train_count_exact":
        len(
            train_df
        )
        ==
        TARGET_TRAIN_COUNT,

    "validation_count_exact":
        len(
            validation_df
        )
        ==
        TARGET_VALIDATION_COUNT,

    "zero_uid_overlap":
        len(
            uid_overlap
        )
        ==
        0,

    "all_master_uids_accounted_for":
        train_validation_union
        ==
        master_uids,

    "train_has_all_classes":
        train_classes
        ==
        master_classes,

    "validation_has_all_classes":
        validation_classes
        ==
        master_classes,

    "train_class_count_215":
        len(
            train_classes
        )
        ==
        EXPECTED_SUPERVISED_CLASSES,

    "validation_class_count_215":
        len(
            validation_classes
        )
        ==
        EXPECTED_SUPERVISED_CLASSES

}


for name, passed in (
    checks.items()
):

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


failed = [

    name

    for name, passed
    in checks.items()

    if not passed

]


if failed:

    raise RuntimeError(
        f"Split integrity failure: {failed}"
    )


# ==================================================================================================
# 19. ACTUAL SPLIT STATISTICS
# ==================================================================================================

actual_train_fraction = (
    len(
        train_df
    )
    /
    len(
        supervised_df
    )
)


actual_validation_fraction = (
    len(
        validation_df
    )
    /
    len(
        supervised_df
    )
)


train_counts = (
    train_df
    .groupby(
        "gloss"
    )
    .size()
)


validation_counts = (
    validation_df
    .groupby(
        "gloss"
    )
    .size()
)


print()

print(
    "TRAIN"
)

print(
    "  Videos:",
    len(
        train_df
    )
)

print(
    "  Classes:",
    train_df[
        "gloss"
    ].nunique()
)

print(
    "  Percentage:",
    f"{actual_train_fraction * 100:.4f}%"
)

print(
    "  Min videos/class:",
    int(
        train_counts.min()
    )
)

print(
    "  Max videos/class:",
    int(
        train_counts.max()
    )
)


print()

print(
    "VALIDATION"
)

print(
    "  Videos:",
    len(
        validation_df
    )
)

print(
    "  Classes:",
    validation_df[
        "gloss"
    ].nunique()
)

print(
    "  Percentage:",
    f"{actual_validation_fraction * 100:.4f}%"
)

print(
    "  Min videos/class:",
    int(
        validation_counts.min()
    )
)

print(
    "  Max videos/class:",
    int(
        validation_counts.max()
    )
)


# ==================================================================================================
# 20. PER-CLASS REPORT
# ==================================================================================================

per_class_rows = []


for gloss in ordered_glosses:

    class_id = int(
        gloss_to_class_id[
            gloss
        ]
    )


    total = int(
        (
            supervised_df[
                "gloss"
            ]
            ==
            gloss
        ).sum()
    )


    n_train = int(
        (
            train_df[
                "gloss"
            ]
            ==
            gloss
        ).sum()
    )


    n_validation = int(
        (
            validation_df[
                "gloss"
            ]
            ==
            gloss
        ).sum()
    )


    per_class_rows.append(
        {

            "class_id":
                class_id,

            "gloss":
                gloss,

            "total_videos":
                total,

            "train_videos":
                n_train,

            "validation_videos":
                n_validation,

            "train_percent":
                (
                    n_train
                    /
                    total
                    *
                    100.0
                ),

            "validation_percent":
                (
                    n_validation
                    /
                    total
                    *
                    100.0
                )

        }
    )


per_class_df = pd.DataFrame(
    per_class_rows
)


per_class_df.to_csv(
    PER_CLASS_SPLIT_PATH,
    index=False
)


# ==================================================================================================
# 21. SAVE SUPERVISED MANIFESTS
# ==================================================================================================

supervised_master_out = (
    supervised_df
    .copy()
)


supervised_master_out[
    "split"
] = ""


split_lookup = {

    uid:
        "train"

    for uid in train_df[
        "uid"
    ]

}


split_lookup.update(
    {

        uid:
            "validation"

        for uid in validation_df[
            "uid"
        ]

    }
)


supervised_master_out[
    "split"
] = (
    supervised_master_out[
        "uid"
    ]
    .map(
        split_lookup
    )
)


supervised_master_out = (
    supervised_master_out
    .sort_values(
        [
            "class_id",
            "uid"
        ]
    )
    .reset_index(
        drop=True
    )
)


supervised_master_out.to_csv(
    SUPERVISED_MASTER_PATH,
    index=False
)


train_df.to_csv(
    TRAIN_MANIFEST_PATH,
    index=False
)


validation_df.to_csv(
    VALIDATION_MANIFEST_PATH,
    index=False
)


print(
    "\n[PASS] Frozen manifests saved."
)


# ==================================================================================================
# 22. FREEZE SPLIT CONFIGURATION
# ==================================================================================================

split_config = {

    "stage":
        "L20B",

    "status":
        "FROZEN",

    "seed":
        SEED,

    "support_threshold":
        SUPPORT_THRESHOLD,

    "num_classes":
        EXPECTED_SUPERVISED_CLASSES,

    "total_supervised_videos":
        EXPECTED_SUPERVISED_VIDEOS,

    "target_train_fraction":
        TARGET_TRAIN_FRACTION,

    "target_validation_fraction":
        TARGET_VALIDATION_FRACTION,

    "actual_train_videos":
        int(
            len(
                train_df
            )
        ),

    "actual_validation_videos":
        int(
            len(
                validation_df
            )
        ),

    "actual_train_fraction":
        float(
            actual_train_fraction
        ),

    "actual_validation_fraction":
        float(
            actual_validation_fraction
        ),

    "split_method":
        (
            "deterministic class-stratified integer allocation "
            "with global 70/30 target"
        ),

    "constraints": {

        "minimum_train_per_class":
            1,

        "minimum_validation_per_class":
            1,

        "uid_overlap_allowed":
            False,

        "seed_search_allowed":
            False,

        "accuracy_based_split_selection":
            False

    }

}


save_json(
    split_config,
    SPLIT_CONFIG_PATH
)


# ==================================================================================================
# 23. DEFINE L20C DOMAIN-PRETRAINING PROTOCOL
# ==================================================================================================
#
# IMPORTANT:
#
# This is only a configuration declaration.
# L20B does NOT execute pretraining.
#
# The 7048-video population will be used label-agnostically/domain-adaptively.
#
# ==================================================================================================

pretrain_config = {

    "next_stage":
        "L20C",

    "task":
        "full_cislr_videomae_domain_pretraining",

    "population_manifest":
        str(
            FULL_PRETRAIN_MANIFEST_PATH
        ),

    "num_videos":
        int(
            len(
                pretrain_df
            )
        ),

    "num_glosses_present":
        int(
            pretrain_df[
                "gloss"
            ].nunique()
        ),

    "use_gloss_labels_for_supervised_classifier":
        False,

    "intended_learning":
        (
            "label-agnostic/self-supervised CISLR video domain adaptation"
        ),

    "backbone_family":
        "VideoMAE",

    "initial_checkpoint":
        "MCG-NJU/videomae-base-finetuned-kinetics",

    "input_frames":
        16,

    "input_size":
        224,

    "seed":
        SEED,

    "executed_in_l20b":
        False

}


save_json(
    pretrain_config,
    PRETRAIN_CONFIG_PATH
)


# ==================================================================================================
# 24. DEFINE L20D SUPERVISED TRAINING PROTOCOL
# ==================================================================================================

supervised_config = {

    "future_stage":
        "L20D",

    "task":
        "215_class_cislr_supervised_video_classification",

    "support_threshold":
        SUPPORT_THRESHOLD,

    "num_classes":
        EXPECTED_SUPERVISED_CLASSES,

    "total_videos":
        EXPECTED_SUPERVISED_VIDEOS,

    "train_manifest":
        str(
            TRAIN_MANIFEST_PATH
        ),

    "validation_manifest":
        str(
            VALIDATION_MANIFEST_PATH
        ),

    "class_mapping":
        str(
            CLASS_MAPPING_PATH
        ),

    "train_videos":
        int(
            len(
                train_df
            )
        ),

    "validation_videos":
        int(
            len(
                validation_df
            )
        ),

    "target_metric":
        "validation_top1_accuracy",

    "requested_target_percent":
        90.0,

    "target_is_guaranteed":
        False,

    "validation_used_for":
        (
            "model selection and early stopping"
        ),

    "validation_used_for_training":
        False,

    "split_search_using_accuracy":
        False,

    "seed_search_using_accuracy":
        False,

    "executed_in_l20b":
        False

}


save_json(
    supervised_config,
    SUPERVISED_CONFIG_PATH
)


# ==================================================================================================
# 25. SUMMARY
# ==================================================================================================

summary = {

    "stage":
        "L20B",

    "status":
        "COMPLETE_AND_FROZEN",

    "seed":
        SEED,

    "full_cislr_population": {

        "videos":
            int(
                len(
                    pretrain_df
                )
            ),

        "classes_present":
            int(
                pretrain_df[
                    "gloss"
                ].nunique()
            ),

        "role":
            (
                "future label-agnostic "
                "VideoMAE domain pretraining"
            )

    },

    "supervised_population": {

        "minimum_support":
            SUPPORT_THRESHOLD,

        "classes":
            int(
                supervised_df[
                    "gloss"
                ].nunique()
            ),

        "videos":
            int(
                len(
                    supervised_df
                )
            ),

        "train_videos":
            int(
                len(
                    train_df
                )
            ),

        "validation_videos":
            int(
                len(
                    validation_df
                )
            ),

        "train_percent":
            float(
                actual_train_fraction *
                100.0
            ),

        "validation_percent":
            float(
                actual_validation_fraction *
                100.0
            )

    },

    "integrity": {

        "uid_overlap":
            int(
                len(
                    uid_overlap
                )
            ),

        "train_classes":
            int(
                train_df[
                    "gloss"
                ].nunique()
            ),

        "validation_classes":
            int(
                validation_df[
                    "gloss"
                ].nunique()
            ),

        "all_supervised_videos_accounted_for":
            bool(
                train_validation_union
                ==
                master_uids
            )

    },

    "model_training_performed":
        False,

    "model_inference_performed":
        False,

    "accuracy_used_to_choose_split":
        False,

    "next_stage":
        "L20C_FULL_CISLR_DOMAIN_PRETRAINING",

    "created_at":
        datetime.now().isoformat()

}


save_json(
    summary,
    SUMMARY_PATH
)


# ==================================================================================================
# 26. HASH CRITICAL ARTIFACTS
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "11. HASH CRITICAL L20B ARTIFACTS"
)

print(
    "=" * 150
)


critical_artifacts = {

    "full_pretraining_manifest":
        FULL_PRETRAIN_MANIFEST_PATH,

    "supervised_master":
        SUPERVISED_MASTER_PATH,

    "train_manifest":
        TRAIN_MANIFEST_PATH,

    "validation_manifest":
        VALIDATION_MANIFEST_PATH,

    "class_mapping":
        CLASS_MAPPING_PATH,

    "split_config":
        SPLIT_CONFIG_PATH,

    "pretraining_config":
        PRETRAIN_CONFIG_PATH,

    "supervised_config":
        SUPERVISED_CONFIG_PATH,

    "per_class_split":
        PER_CLASS_SPLIT_PATH,

    "summary":
        SUMMARY_PATH

}


hash_manifest = {}


for name, path in (
    critical_artifacts.items()
):

    digest = sha256_file(
        path
    )


    hash_manifest[
        name
    ] = {

        "path":
            str(
                path
            ),

        "sha256":
            digest

    }


    print(
        f"{name:30s}: {digest}"
    )


save_json(
    hash_manifest,
    HASH_MANIFEST_PATH
)


# ==================================================================================================
# 27. MASTER FREEZE
# ==================================================================================================

master_freeze = {

    "stage":
        "L20B",

    "status":
        "COMPLETE_AND_FROZEN",

    "seed":
        SEED,

    "decision_frozen_before_training":
        True,

    "selected_supervised_design": {

        "support_threshold":
            SUPPORT_THRESHOLD,

        "classes":
            int(
                len(
                    ordered_glosses
                )
            ),

        "videos":
            int(
                len(
                    supervised_df
                )
            ),

        "train_videos":
            int(
                len(
                    train_df
                )
            ),

        "validation_videos":
            int(
                len(
                    validation_df
                )
            ),

        "train_fraction":
            float(
                actual_train_fraction
            ),

        "validation_fraction":
            float(
                actual_validation_fraction
            )

    },

    "full_domain_pretraining_design": {

        "videos":
            int(
                len(
                    pretrain_df
                )
            ),

        "classes_present":
            int(
                pretrain_df[
                    "gloss"
                ].nunique()
            ),

        "labels_used_for_classifier":
            False

    },

    "forbidden_after_freeze": [

        "change support threshold because of validation accuracy",

        "search random seeds for better validation accuracy",

        "move validation videos into training",

        "duplicate validation videos into training",

        "select split based on model performance"

    ],

    "next_stage":
        "L20C_FULL_CISLR_DOMAIN_PRETRAINING",

    "created_at":
        datetime.now().isoformat()

}


save_json(
    master_freeze,
    FREEZE_PATH
)


MASTER_SHA256 = sha256_file(
    FREEZE_PATH
)


# ==================================================================================================
# 28. FINAL INTEGRITY CHECK
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "12. FINAL L20B INTEGRITY CHECK"
)

print(
    "=" * 150
)


final_checks = {

    "full_population_7048":
        len(
            pretrain_df
        )
        ==
        EXPECTED_FULL_CLEAN_VIDEOS,

    "full_population_4763_classes":
        pretrain_df[
            "gloss"
        ].nunique()
        ==
        EXPECTED_FULL_CLASSES,

    "supervised_population_1024":
        len(
            supervised_df
        )
        ==
        EXPECTED_SUPERVISED_VIDEOS,

    "supervised_classes_215":
        supervised_df[
            "gloss"
        ].nunique()
        ==
        EXPECTED_SUPERVISED_CLASSES,

    "train_target_reached":
        len(
            train_df
        )
        ==
        TARGET_TRAIN_COUNT,

    "validation_target_reached":
        len(
            validation_df
        )
        ==
        TARGET_VALIDATION_COUNT,

    "zero_train_validation_overlap":
        len(
            uid_overlap
        )
        ==
        0,

    "train_has_all_215_classes":
        train_df[
            "gloss"
        ].nunique()
        ==
        EXPECTED_SUPERVISED_CLASSES,

    "validation_has_all_215_classes":
        validation_df[
            "gloss"
        ].nunique()
        ==
        EXPECTED_SUPERVISED_CLASSES,

    "all_supervised_videos_accounted":
        train_validation_union
        ==
        master_uids,

    "class_mapping_saved":
        CLASS_MAPPING_PATH.exists(),

    "train_manifest_saved":
        TRAIN_MANIFEST_PATH.exists(),

    "validation_manifest_saved":
        VALIDATION_MANIFEST_PATH.exists(),

    "pretraining_manifest_saved":
        FULL_PRETRAIN_MANIFEST_PATH.exists(),

    "no_model_training":
        True,

    "no_model_inference":
        True,

    "accuracy_not_used_for_split":
        True,

    "freeze_saved":
        FREEZE_PATH.exists()

}


for name, passed in (
    final_checks.items()
):

    print(
        f"{name:45s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


failed_checks = [

    name

    for name, passed
    in final_checks.items()

    if not passed

]


if failed_checks:

    raise RuntimeError(

        "L20B integrity failure: "
        f"{failed_checks}"

    )


# ==================================================================================================
# 29. FINAL REPORT
# ==================================================================================================

print(
    "\n"
)

print(
    "=" * 150
)

print(
    "L20B — FULL-CISLR LEARNING PROTOCOL FROZEN"
)

print(
    "=" * 150
)

print()


print(
    "FULL CISLR DOMAIN-PRETRAINING POPULATION"
)

print(
    f"  Videos                    : {len(pretrain_df)}"
)

print(
    f"  Glosses present           : {pretrain_df['gloss'].nunique()}"
)

print(
    "  Role                      : future label-agnostic VideoMAE domain pretraining"
)


print()


print(
    "SUPERVISED CLASSIFICATION POPULATION"
)

print(
    f"  Minimum class support     : >= {SUPPORT_THRESHOLD}"
)

print(
    f"  Classes                   : {supervised_df['gloss'].nunique()}"
)

print(
    f"  Independent videos        : {len(supervised_df)}"
)


print()


print(
    "CORRECTED STRATIFIED SPLIT"
)

print(
    f"  Train videos              : {len(train_df)}"
)

print(
    f"  Validation videos         : {len(validation_df)}"
)

print(
    f"  Train percentage          : {actual_train_fraction * 100:.4f}%"
)

print(
    f"  Validation percentage     : {actual_validation_fraction * 100:.4f}%"
)

print(
    f"  Train classes             : {train_df['gloss'].nunique()}"
)

print(
    f"  Validation classes        : {validation_df['gloss'].nunique()}"
)

print(
    f"  UID overlap               : {len(uid_overlap)}"
)


print()


print(
    "TRAIN PER-CLASS"
)

print(
    f"  Minimum                   : {int(train_counts.min())}"
)

print(
    f"  Maximum                   : {int(train_counts.max())}"
)


print()


print(
    "VALIDATION PER-CLASS"
)

print(
    f"  Minimum                   : {int(validation_counts.min())}"
)

print(
    f"  Maximum                   : {int(validation_counts.max())}"
)


print()


print(
    "ARTIFACTS"
)


print(
    "  Full 7048-video pretraining manifest:"
)

print(
    f"    {FULL_PRETRAIN_MANIFEST_PATH}"
)


print()


print(
    "  Supervised master:"
)

print(
    f"    {SUPERVISED_MASTER_PATH}"
)


print()


print(
    "  Training manifest:"
)

print(
    f"    {TRAIN_MANIFEST_PATH}"
)


print()


print(
    "  Validation manifest:"
)

print(
    f"    {VALIDATION_MANIFEST_PATH}"
)


print()


print(
    "  215-class mapping:"
)

print(
    f"    {CLASS_MAPPING_PATH}"
)


print()


print(
    "  Per-class split report:"
)

print(
    f"    {PER_CLASS_SPLIT_PATH}"
)


print()


print(
    "  Master freeze:"
)

print(
    f"    {FREEZE_PATH}"
)


print()


print(
    "L20B MASTER SHA256:"
)

print(
    f"  {MASTER_SHA256}"
)


print()


print(
    "[PASS] L20B COMPLETE AND FROZEN"
)

print(
    "[PASS] SUPPORT >= 4 DESIGN FROZEN BEFORE TRAINING"
)

print(
    "[PASS] 215-CLASS MAPPING FROZEN"
)

print(
    "[PASS] GLOBAL ~70/30 SPLIT FROZEN"
)

print(
    "[PASS] EVERY CLASS EXISTS IN TRAIN AND VALIDATION"
)

print(
    "[PASS] ZERO TRAIN/VALIDATION UID OVERLAP"
)

print(
    "[PASS] ALL 7048 CLEAN CISLR VIDEOS RESERVED FOR DOMAIN PRETRAINING"
)

print(
    "[PASS] NO MODEL TRAINING OR INFERENCE PERFORMED"
)

print(
    "[PASS] NO ACCURACY-BASED SPLIT SELECTION"
)

print()


print(
    "[LOCK] DO NOT CHANGE THIS SPLIT BASED ON LATER VALIDATION ACCURACY"
)

print()


print(
    "[NEXT] L20C = FULL 7048-VIDEO CISLR VIDEOMAE DOMAIN PRETRAINING"
)

print(
    "=" * 150
)

CISLR — L20B
FULL-CISLR LEARNING PROTOCOL FREEZE
SUPPORT >= 4 | 215-CLASS SUPERVISED DATASET | CORRECTED NEAR-70/30 SPLIT

Python: 3.11.15
Pandas: 3.0.5
NumPy: 2.4.6
Seed: 42
Frozen support threshold: 4
Target train fraction: 0.7
Target validation fraction: 0.3

1. VERIFY L20A INPUT ARTIFACTS
L20A clean manifest                : PASS
L20A support table                 : PASS
L20A threshold summary             : PASS
L20A master freeze                 : PASS

2. VERIFY L20A FREEZE
L20A stage: L20A
L20A status: COMPLETE_AND_FROZEN
[PASS] L20A freeze verified.

3. LOAD FULL CLEAN CISLR POPULATION
Clean videos: 7048
Unique classes: 4763
[PASS] Full L20A population matches frozen audit.

4. CREATE FULL 7048-VIDEO DOMAIN-PRETRAINING MANIFEST
Domain-pretraining videos: 7048
Domain-pretraining gloss population: 4763
[PASS] All 7048 clean CISLR videos assigned to domain-pretraining population.

5. RECOMPUTE CLASS SUPPORT
Support >= 4 classes: 215
Minimum retained support: 4
Maximum retained sup

In [2]:
# ==================================================================================================
# CISLR — L20C
#
# FULL CISLR VIDEOMAE DOMAIN ADAPTATION
# VALIDATION-EXCLUDED / RESUME-SAFE
#
# --------------------------------------------------------------------------------------------------
# FROZEN INPUT FROM L20B
# --------------------------------------------------------------------------------------------------
#
# Full clean CISLR population : 7048 videos
# Supervised task             : 215 classes / 1024 videos
# Supervised train            : 717 videos
# Frozen validation           : 307 videos
#
# --------------------------------------------------------------------------------------------------
# L20C POLICY
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
#
# The 307 L20B validation videos are EXCLUDED from L20C.
#
# Therefore:
#
#     7048 - 307 = 6741
#
# videos are eligible for CISLR domain adaptation.
#
# Validation videos are:
#
#     NOT decoded
#     NOT cached
#     NOT trained on
#     NOT used for early stopping
#     NOT used for hyperparameter selection
#
# --------------------------------------------------------------------------------------------------
# METHOD
# --------------------------------------------------------------------------------------------------
#
# Pretrained VideoMAE backbone:
#
#     MCG-NJU/videomae-base-finetuned-kinetics
#
# For every training video:
#
#     view A = one temporally sampled 16-frame clip
#     view B = another temporally perturbed 16-frame clip
#
# Both views pass through VideoMAE.
#
# We optimize:
#
#     1. cosine representation consistency
#     2. representation variance regularization
#
# This adapts the pretrained visual representation to CISLR videos
# without using gloss labels.
#
# NOTE:
# This is label-agnostic domain adaptation / representation learning.
# It is NOT the original masked-pixel VideoMAE pretraining objective.
#
# L20D will use the resulting backbone for the frozen 215-class task.
#
# --------------------------------------------------------------------------------------------------
# HARDWARE
# --------------------------------------------------------------------------------------------------
#
# Designed for:
#
#     RTX 4050 Laptop GPU (~6 GB)
#
# Conservative defaults:
#
#     batch size          = 1
#     gradient accumulation = 8
#     mixed precision     = True
#     final 4 transformer blocks trainable
#
# --------------------------------------------------------------------------------------------------
# RESUME
# --------------------------------------------------------------------------------------------------
#
# The cell saves:
#
#     best checkpoint
#     last/resume checkpoint
#     epoch history
#
# Re-running the cell automatically resumes from the last completed epoch.
#
# ==================================================================================================


# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

import os
import gc
import cv2
import json
import math
import time
import random
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

from transformers import (
    VideoMAEForVideoClassification
)


warnings.filterwarnings(
    "ignore"
)


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

SEED = 42


def seed_everything(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


seed_everything(
    SEED
)


# ==================================================================================================
# 2. TORCH PERFORMANCE SETTINGS
# ==================================================================================================

if torch.cuda.is_available():

    torch.backends.cudnn.benchmark = True

    try:

        torch.set_float32_matmul_precision(
            "high"
        )

    except Exception:

        pass


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else
    "cpu"
)


print(
    "=" * 150
)

print(
    "CISLR — L20C"
)

print(
    "VALIDATION-EXCLUDED VIDEOMAE DOMAIN ADAPTATION"
)

print(
    "=" * 150
)

print()

print(
    "Python:",
    platform.python_version()
)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "OpenCV:",
    cv2.__version__
)

print(
    "Device:",
    DEVICE
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )


print()


# ==================================================================================================
# 3. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)


ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)


L20B_ROOT = (
    ROOT /
    "audit/l20b_full_cislr_protocol_freeze"
)


FULL_MANIFEST_PATH = (
    L20B_ROOT /
    "manifests/l20b_full_7048_domain_pretraining_manifest.csv"
)


TRAIN_MANIFEST_PATH = (
    L20B_ROOT /
    "manifests/l20b_train_manifest.csv"
)


VALIDATION_MANIFEST_PATH = (
    L20B_ROOT /
    "manifests/l20b_validation_manifest.csv"
)


L20B_FREEZE_PATH = (
    L20B_ROOT /
    "L20B_MASTER_FREEZE.json"
)


VIDEO_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_v1.5-a_videos"
)


# --------------------------------------------------------------------------------------------------
# L20C outputs
# --------------------------------------------------------------------------------------------------

L20C_ROOT = (
    ROOT /
    "audit/l20c_full_cislr_domain_adaptation"
)


MODEL_DIR = (
    L20C_ROOT /
    "models"
)


REPORT_DIR = (
    L20C_ROOT /
    "reports"
)


MANIFEST_DIR = (
    L20C_ROOT /
    "manifests"
)


HASH_DIR = (
    L20C_ROOT /
    "hashes"
)


CONFIG_DIR = (
    L20C_ROOT /
    "config"
)


for directory in [

    L20C_ROOT,
    MODEL_DIR,
    REPORT_DIR,
    MANIFEST_DIR,
    HASH_DIR,
    CONFIG_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


ELIGIBLE_MANIFEST_PATH = (
    MANIFEST_DIR /
    "l20c_domain_adaptation_6741_manifest.csv"
)


EXCLUDED_VALIDATION_PATH = (
    MANIFEST_DIR /
    "l20c_excluded_validation_manifest.csv"
)


BEST_MODEL_PATH = (
    MODEL_DIR /
    "l20c_best_cislr_adapted_videomae.pt"
)


LAST_MODEL_PATH = (
    MODEL_DIR /
    "l20c_last_resume_checkpoint.pt"
)


BACKBONE_PATH = (
    MODEL_DIR /
    "l20c_final_cislr_adapted_backbone.pt"
)


HISTORY_PATH = (
    REPORT_DIR /
    "l20c_training_history.csv"
)


SUMMARY_PATH = (
    REPORT_DIR /
    "l20c_summary.json"
)


CONFIG_PATH = (
    CONFIG_DIR /
    "l20c_config.json"
)


HASH_PATH = (
    HASH_DIR /
    "l20c_sha256_manifest.json"
)


FREEZE_PATH = (
    L20C_ROOT /
    "L20C_MASTER_FREEZE.json"
)


# ==================================================================================================
# 4. TRAINING CONFIGURATION
# ==================================================================================================

BASE_MODEL = (
    "MCG-NJU/videomae-base-finetuned-kinetics"
)


NUM_FRAMES = 16

INPUT_SIZE = 224


# --------------------------------------------------------------------------------------------------
# GPU-safe defaults
# --------------------------------------------------------------------------------------------------

BATCH_SIZE = 1

GRAD_ACCUM_STEPS = 8

NUM_WORKERS = 0


# --------------------------------------------------------------------------------------------------
# Training
# --------------------------------------------------------------------------------------------------

MAX_EPOCHS = 15

LEARNING_RATE = 1e-5

WEIGHT_DECAY = 1e-4

WARMUP_EPOCHS = 1

MIN_LR_RATIO = 0.05

GRAD_CLIP = 1.0


# --------------------------------------------------------------------------------------------------
# Backbone adaptation depth
# --------------------------------------------------------------------------------------------------

TRAINABLE_FINAL_BLOCKS = 4


# --------------------------------------------------------------------------------------------------
# Loss
# --------------------------------------------------------------------------------------------------

CONSISTENCY_WEIGHT = 1.0

VARIANCE_WEIGHT = 0.10

VARIANCE_TARGET_STD = 1.0


# --------------------------------------------------------------------------------------------------
# Safety / logging
# --------------------------------------------------------------------------------------------------

LOG_EVERY = 250

SAVE_EVERY_EPOCH = True


# ==================================================================================================
# 5. IMAGE NORMALIZATION
# ==================================================================================================
#
# Same normalization convention used by the previous VideoMAE experiment.
#
# ==================================================================================================

IMAGE_MEAN = torch.tensor(
    [
        0.485,
        0.456,
        0.406
    ],
    dtype=torch.float32
).view(
    1,
    3,
    1,
    1
)


IMAGE_STD = torch.tensor(
    [
        0.229,
        0.224,
        0.225
    ],
    dtype=torch.float32
).view(
    1,
    3,
    1,
    1
)


# ==================================================================================================
# 6. HELPERS
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:

                break

            h.update(
                block
            )

    return h.hexdigest()


def save_json(
    obj,
    path
):

    with open(
        path,
        "w"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )


def normalize_uid(value):

    uid = str(
        value
    ).strip()

    if uid.lower().endswith(
        ".mp4"
    ):

        uid = uid[:-4]

    return uid


# ==================================================================================================
# 7. VERIFY L20B
# ==================================================================================================

print(
    "=" * 150
)

print(
    "1. VERIFY FROZEN L20B"
)

print(
    "=" * 150
)


required = {

    "full manifest":
        FULL_MANIFEST_PATH,

    "train manifest":
        TRAIN_MANIFEST_PATH,

    "validation manifest":
        VALIDATION_MANIFEST_PATH,

    "L20B freeze":
        L20B_FREEZE_PATH

}


for name, path in (
    required.items()
):

    exists = path.exists()

    print(
        f"{name:30s}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    if not exists:

        raise FileNotFoundError(
            path
        )


with open(
    L20B_FREEZE_PATH,
    "r"
) as f:

    l20b_freeze = json.load(
        f
    )


if (
    l20b_freeze.get(
        "stage"
    )
    !=
    "L20B"
):

    raise RuntimeError(
        "Wrong L20B freeze stage."
    )


if (
    l20b_freeze.get(
        "status"
    )
    !=
    "COMPLETE_AND_FROZEN"
):

    raise RuntimeError(
        "L20B is not frozen."
    )


print(
    "[PASS] L20B freeze verified."
)


# ==================================================================================================
# 8. LOAD MANIFESTS
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "2. BUILD VALIDATION-EXCLUDED L20C POPULATION"
)

print(
    "=" * 150
)


full_df = pd.read_csv(
    FULL_MANIFEST_PATH
)


train_df = pd.read_csv(
    TRAIN_MANIFEST_PATH
)


validation_df = pd.read_csv(
    VALIDATION_MANIFEST_PATH
)


for frame in [

    full_df,
    train_df,
    validation_df

]:

    frame[
        "uid"
    ] = (
        frame[
            "uid"
        ]
        .apply(
            normalize_uid
        )
    )


print(
    "Full clean population:",
    len(
        full_df
    )
)


print(
    "L20B supervised train:",
    len(
        train_df
    )
)


print(
    "Frozen validation:",
    len(
        validation_df
    )
)


if len(
    full_df
) != 7048:

    raise RuntimeError(
        f"Expected 7048 full videos; found {len(full_df)}."
    )


if len(
    train_df
) != 717:

    raise RuntimeError(
        f"Expected 717 supervised train videos; found {len(train_df)}."
    )


if len(
    validation_df
) != 307:

    raise RuntimeError(
        f"Expected 307 validation videos; found {len(validation_df)}."
    )


validation_uids = set(
    validation_df[
        "uid"
    ]
)


eligible_df = full_df[
    ~full_df[
        "uid"
    ].isin(
        validation_uids
    )
].copy()


eligible_df = (
    eligible_df
    .sort_values(
        "uid"
    )
    .reset_index(
        drop=True
    )
)


print()

print(
    "L20C eligible videos:",
    len(
        eligible_df
    )
)


print(
    "Expected:",
    7048 - 307
)


if len(
    eligible_df
) != 6741:

    raise RuntimeError(
        f"Expected 6741 L20C videos; found {len(eligible_df)}."
    )


overlap = (
    set(
        eligible_df[
            "uid"
        ]
    )
    &
    validation_uids
)


print(
    "L20C ↔ validation UID overlap:",
    len(
        overlap
    )
)


if overlap:

    raise RuntimeError(
        "VALIDATION LEAKAGE DETECTED."
    )


eligible_df.to_csv(
    ELIGIBLE_MANIFEST_PATH,
    index=False
)


validation_df.to_csv(
    EXCLUDED_VALIDATION_PATH,
    index=False
)


print(
    "[PASS] 307 validation videos excluded from L20C."
)


# ==================================================================================================
# 9. VERIFY VIDEO PATHS
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "3. VERIFY L20C VIDEO PATHS"
)

print(
    "=" * 150
)


# L20A/L20B should already contain video_path.
# Fall back to VIDEO_ROOT/uid.mp4 only if necessary.

resolved_paths = []

missing_uids = []


for _, row in (
    eligible_df.iterrows()
):

    uid = row[
        "uid"
    ]


    candidate = None


    if (
        "video_path"
        in
        eligible_df.columns
    ):

        value = row.get(
            "video_path",
            ""
        )

        if pd.notna(
            value
        ):

            p = Path(
                str(
                    value
                )
            )

            if p.exists():

                candidate = p


    if candidate is None:

        p = (
            VIDEO_ROOT /
            f"{uid}.mp4"
        )

        if p.exists():

            candidate = p


    if candidate is None:

        # recursive fallback only when direct lookup fails

        matches = list(
            VIDEO_ROOT.rglob(
                f"{uid}.mp4"
            )
        )

        if matches:

            candidate = matches[
                0
            ]


    if candidate is None:

        resolved_paths.append(
            ""
        )

        missing_uids.append(
            uid
        )

    else:

        resolved_paths.append(
            str(
                candidate
            )
        )


eligible_df[
    "resolved_video_path"
] = resolved_paths


print(
    "Resolved:",
    len(
        eligible_df
    )
    -
    len(
        missing_uids
    )
)


print(
    "Missing:",
    len(
        missing_uids
    )
)


if missing_uids:

    print(
        "First missing UIDs:",
        missing_uids[
            :20
        ]
    )

    raise RuntimeError(
        "Some L20C videos are missing."
    )


eligible_df.to_csv(
    ELIGIBLE_MANIFEST_PATH,
    index=False
)


# ==================================================================================================
# 10. SAVE CONFIG BEFORE TRAINING
# ==================================================================================================

config = {

    "stage":
        "L20C",

    "task":
        (
            "validation_excluded_cislr_videomae_"
            "label_agnostic_domain_adaptation"
        ),

    "seed":
        SEED,

    "base_model":
        BASE_MODEL,

    "full_population":
        7048,

    "excluded_validation":
        307,

    "domain_adaptation_population":
        int(
            len(
                eligible_df
            )
        ),

    "validation_uid_overlap":
        int(
            len(
                overlap
            )
        ),

    "num_frames":
        NUM_FRAMES,

    "input_size":
        INPUT_SIZE,

    "batch_size":
        BATCH_SIZE,

    "gradient_accumulation_steps":
        GRAD_ACCUM_STEPS,

    "effective_batch_size":
        (
            BATCH_SIZE
            *
            GRAD_ACCUM_STEPS
        ),

    "epochs":
        MAX_EPOCHS,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "trainable_final_transformer_blocks":
        TRAINABLE_FINAL_BLOCKS,

    "loss": {

        "cosine_consistency_weight":
            CONSISTENCY_WEIGHT,

        "variance_weight":
            VARIANCE_WEIGHT,

        "variance_target_std":
            VARIANCE_TARGET_STD

    },

    "validation_frames_loaded":
        False,

    "gloss_labels_used_for_training":
        False,

    "resume_safe":
        True,

    "created_at":
        datetime.now().isoformat()

}


save_json(
    config,
    CONFIG_PATH
)


# ==================================================================================================
# 11. ROBUST SEQUENTIAL VIDEO DECODER
# ==================================================================================================

def decode_video_sequentially(
    video_path
):

    cap = cv2.VideoCapture(
        str(
            video_path
        )
    )


    if not cap.isOpened():

        raise RuntimeError(
            f"Cannot open video: {video_path}"
        )


    frames = []


    while True:

        ok, frame = cap.read()

        if not ok:

            break


        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )


        frames.append(
            frame
        )


    cap.release()


    if len(
        frames
    ) == 0:

        raise RuntimeError(
            f"No frames decoded: {video_path}"
        )


    return frames


# ==================================================================================================
# 12. TEMPORAL VIEW GENERATION
# ==================================================================================================

def sample_temporal_indices(
    total_frames,
    num_frames,
    rng
):

    if total_frames <= 0:

        raise ValueError(
            "Video has zero frames."
        )


    # ----------------------------------------------------------------------------------------------
    # Short video:
    # sample with interpolation-like repeated indices.
    # ----------------------------------------------------------------------------------------------

    if total_frames <= num_frames:

        indices = np.linspace(
            0,
            total_frames - 1,
            num_frames
        )

        return np.round(
            indices
        ).astype(
            np.int64
        )


    # ----------------------------------------------------------------------------------------------
    # Temporal bin jitter:
    #
    # Divide entire video into NUM_FRAMES temporal bins.
    # Randomly choose one frame per bin.
    #
    # This preserves full-sign temporal coverage while generating
    # different views.
    # ----------------------------------------------------------------------------------------------

    boundaries = np.linspace(
        0,
        total_frames,
        num_frames + 1
    )


    indices = []


    for i in range(
        num_frames
    ):

        low = int(
            math.floor(
                boundaries[
                    i
                ]
            )
        )


        high = int(
            math.ceil(
                boundaries[
                    i + 1
                ]
            )
        )


        low = max(
            0,
            min(
                low,
                total_frames - 1
            )
        )


        high = max(
            low + 1,
            min(
                high,
                total_frames
            )
        )


        idx = int(
            rng.integers(
                low,
                high
            )
        )


        indices.append(
            idx
        )


    return np.asarray(
        indices,
        dtype=np.int64
    )


# ==================================================================================================
# 13. SPATIAL AUGMENTATION
# ==================================================================================================

def resize_short_side(
    clip,
    short_side=256
):

    # clip: [T,C,H,W]

    _, _, h, w = clip.shape


    if h <= 0 or w <= 0:

        raise RuntimeError(
            "Invalid frame dimensions."
        )


    if h < w:

        new_h = short_side

        new_w = int(
            round(
                w *
                short_side /
                h
            )
        )

    else:

        new_w = short_side

        new_h = int(
            round(
                h *
                short_side /
                w
            )
        )


    clip = F.interpolate(
        clip,
        size=(
            new_h,
            new_w
        ),
        mode="bilinear",
        align_corners=False
    )


    return clip


def random_crop_clip(
    clip,
    size,
    rng
):

    _, _, h, w = clip.shape


    if h < size or w < size:

        clip = F.interpolate(
            clip,
            size=(
                max(
                    h,
                    size
                ),
                max(
                    w,
                    size
                )
            ),
            mode="bilinear",
            align_corners=False
        )


        _, _, h, w = clip.shape


    top = int(
        rng.integers(
            0,
            h - size + 1
        )
    )


    left = int(
        rng.integers(
            0,
            w - size + 1
        )
    )


    return clip[
        :,
        :,
        top:top + size,
        left:left + size
    ]


def apply_same_color_adjustment(
    clip,
    brightness,
    contrast,
    saturation
):

    # Apply same factors to every frame.

    output = []


    for frame in clip:

        x = TF.adjust_brightness(
            frame,
            brightness
        )


        x = TF.adjust_contrast(
            x,
            contrast
        )


        x = TF.adjust_saturation(
            x,
            saturation
        )


        output.append(
            x
        )


    return torch.stack(
        output,
        dim=0
    )


def augment_clip(
    frames,
    indices,
    rng
):

    selected = [
        frames[
            int(
                idx
            )
        ]
        for idx in indices
    ]


    clip_np = np.stack(
        selected,
        axis=0
    )


    # [T,H,W,C] -> [T,C,H,W]

    clip = torch.from_numpy(
        clip_np
    ).permute(
        0,
        3,
        1,
        2
    ).float() / 255.0


    # ----------------------------------------------------------------------------------------------
    # Resize
    # ----------------------------------------------------------------------------------------------

    resize_target = int(
        rng.integers(
            248,
            281
        )
    )


    clip = resize_short_side(
        clip,
        resize_target
    )


    # ----------------------------------------------------------------------------------------------
    # Random crop
    # ----------------------------------------------------------------------------------------------

    clip = random_crop_clip(
        clip,
        INPUT_SIZE,
        rng
    )


    # ----------------------------------------------------------------------------------------------
    # Mild affine
    # ----------------------------------------------------------------------------------------------

    angle = float(
        rng.uniform(
            -5.0,
            5.0
        )
    )


    translate_x = int(
        rng.uniform(
            -0.04,
            0.04
        )
        *
        INPUT_SIZE
    )


    translate_y = int(
        rng.uniform(
            -0.04,
            0.04
        )
        *
        INPUT_SIZE
    )


    affine_frames = []


    for frame in clip:

        x = TF.affine(
            frame,
            angle=angle,
            translate=[
                translate_x,
                translate_y
            ],
            scale=1.0,
            shear=[
                0.0,
                0.0
            ],
            interpolation=InterpolationMode.BILINEAR
        )


        affine_frames.append(
            x
        )


    clip = torch.stack(
        affine_frames,
        dim=0
    )


    # ----------------------------------------------------------------------------------------------
    # Color jitter
    # ----------------------------------------------------------------------------------------------

    brightness = float(
        rng.uniform(
            0.85,
            1.15
        )
    )


    contrast = float(
        rng.uniform(
            0.85,
            1.15
        )
    )


    saturation = float(
        rng.uniform(
            0.85,
            1.15
        )
    )


    clip = apply_same_color_adjustment(
        clip,
        brightness,
        contrast,
        saturation
    )


    clip = torch.clamp(
        clip,
        0.0,
        1.0
    )


    # ----------------------------------------------------------------------------------------------
    # Normalize
    # ----------------------------------------------------------------------------------------------

    mean = IMAGE_MEAN.to(
        clip.dtype
    )


    std = IMAGE_STD.to(
        clip.dtype
    )


    clip = (
        clip
        -
        mean
    ) / std


    return clip


# ==================================================================================================
# 14. DATASET
# ==================================================================================================

class CISLRDomainAdaptationDataset(
    Dataset
):

    def __init__(
        self,
        manifest,
        seed
    ):

        self.manifest = (
            manifest
            .reset_index(
                drop=True
            )
            .copy()
        )


        self.seed = int(
            seed
        )


        self.epoch = 0


    def set_epoch(
        self,
        epoch
    ):

        self.epoch = int(
            epoch
        )


    def __len__(
        self
    ):

        return len(
            self.manifest
        )


    def __getitem__(
        self,
        index
    ):

        row = self.manifest.iloc[
            index
        ]


        uid = str(
            row[
                "uid"
            ]
        )


        video_path = Path(
            row[
                "resolved_video_path"
            ]
        )


        # ------------------------------------------------------------------------------------------
        # Deterministic per-sample/per-epoch randomness.
        # Two different view seeds.
        # ------------------------------------------------------------------------------------------

        seed_a = (
            self.seed
            +
            self.epoch * 1000003
            +
            index * 37
            +
            11
        )


        seed_b = (
            self.seed
            +
            self.epoch * 1000003
            +
            index * 37
            +
            97
        )


        rng_a = np.random.default_rng(
            seed_a
        )


        rng_b = np.random.default_rng(
            seed_b
        )


        frames = decode_video_sequentially(
            video_path
        )


        total_frames = len(
            frames
        )


        indices_a = sample_temporal_indices(
            total_frames,
            NUM_FRAMES,
            rng_a
        )


        indices_b = sample_temporal_indices(
            total_frames,
            NUM_FRAMES,
            rng_b
        )


        view_a = augment_clip(
            frames,
            indices_a,
            rng_a
        )


        view_b = augment_clip(
            frames,
            indices_b,
            rng_b
        )


        return {

            "uid":
                uid,

            "view_a":
                view_a,

            "view_b":
                view_b

        }


# ==================================================================================================
# 15. DATASET + LOADER
# ==================================================================================================

dataset = CISLRDomainAdaptationDataset(
    eligible_df,
    seed=SEED
)


loader_generator = torch.Generator()

loader_generator.manual_seed(
    SEED
)


loader = DataLoader(

    dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type
        ==
        "cuda"
    ),

    drop_last=False,

    generator=loader_generator

)


print(
    "\n" +
    "=" * 150
)

print(
    "4. DATA LOADER"
)

print(
    "=" * 150
)

print(
    "Dataset videos:",
    len(
        dataset
    )
)

print(
    "Batches/epoch:",
    len(
        loader
    )
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRAD_ACCUM_STEPS
)

print(
    "Effective batch size:",
    BATCH_SIZE
    *
    GRAD_ACCUM_STEPS
)


# ==================================================================================================
# 16. LOAD VIDEOMAE
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "5. LOAD PRETRAINED VIDEOMAE"
)

print(
    "=" * 150
)


model = VideoMAEForVideoClassification.from_pretrained(
    BASE_MODEL
)


print(
    "Base model loaded:",
    BASE_MODEL
)


# ==================================================================================================
# 17. REMOVE CLASSIFIER FROM REPRESENTATION PATH
# ==================================================================================================
#
# We use:
#
#     model.videomae(...)
#
# rather than model(...)
#
# This gives the transformer hidden representation without using
# the Kinetics classification head.
#
# ==================================================================================================

if not hasattr(
    model,
    "videomae"
):

    raise RuntimeError(
        "Expected Hugging Face VideoMAE model.videomae."
    )


# ==================================================================================================
# 18. FREEZE BACKBONE EXCEPT FINAL N BLOCKS
# ==================================================================================================

for parameter in (
    model.parameters()
):

    parameter.requires_grad = False


encoder_layers = (
    model
    .videomae
    .encoder
    .layer
)


num_encoder_layers = len(
    encoder_layers
)


print(
    "Transformer blocks:",
    num_encoder_layers
)


if (
    TRAINABLE_FINAL_BLOCKS
    >
    num_encoder_layers
):

    raise RuntimeError(
        "TRAINABLE_FINAL_BLOCKS exceeds encoder depth."
    )


for layer in encoder_layers[
    -TRAINABLE_FINAL_BLOCKS:
]:

    for parameter in (
        layer.parameters()
    ):

        parameter.requires_grad = True


# Keep final normalization trainable if available.

# ==================================================================================================
# KEEP FINAL NORMALIZATION TRAINABLE IF IT ACTUALLY EXISTS
# ==================================================================================================
#
# Some Hugging Face VideoMAE configurations define:
#
#     model.videomae.layernorm
#
# but set it to:
#
#     None
#
# Therefore hasattr() alone is not sufficient.
#
# ==================================================================================================

final_layernorm = getattr(
    model.videomae,
    "layernorm",
    None
)


if final_layernorm is not None:

    for parameter in (
        final_layernorm.parameters()
    ):

        parameter.requires_grad = True

    print(
        "[INFO] Final VideoMAE layernorm found and unfrozen."
    )

else:

    print(
        "[INFO] model.videomae.layernorm = None"
    )

    print(
        "[INFO] Skipping final layernorm unfreeze."
    )

    print(
        "[INFO] Final transformer blocks remain trainable."
    )


model = model.to(
    DEVICE
)


total_parameters = sum(
    p.numel()
    for p in model.parameters()
)


trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "Total parameters:",
    f"{total_parameters:,}"
)


print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)


print(
    "Trainable final blocks:",
    TRAINABLE_FINAL_BLOCKS
)


# ==================================================================================================
# 19. REPRESENTATION EXTRACTION
# ==================================================================================================

def get_video_representation(
    pixel_values
):

    outputs = model.videomae(
        pixel_values=pixel_values,
        return_dict=True
    )


    hidden = outputs.last_hidden_state


    # Mean pool all spatiotemporal tokens.

    representation = hidden.mean(
        dim=1
    )


    return representation


# ==================================================================================================
# 20. DOMAIN-ADAPTATION LOSS
# ==================================================================================================

def representation_loss(
    z1,
    z2
):

    # ----------------------------------------------------------------------------------------------
    # Normalize for cosine consistency
    # ----------------------------------------------------------------------------------------------

    z1_norm = F.normalize(
        z1,
        dim=-1
    )


    z2_norm = F.normalize(
        z2,
        dim=-1
    )


    cosine_similarity = (
        z1_norm
        *
        z2_norm
    ).sum(
        dim=-1
    )


    consistency_loss = (
        1.0
        -
        cosine_similarity
    ).mean()


    # ----------------------------------------------------------------------------------------------
    # Variance regularization
    #
    # With BATCH_SIZE=1, the effective mini-batch itself cannot provide
    # a meaningful across-sample variance estimate.
    #
    # Therefore variance is computed across the two views and feature
    # dimensions in a conservative scalar form.
    # ----------------------------------------------------------------------------------------------

    combined = torch.cat(
        [
            z1,
            z2
        ],
        dim=0
    )


    std = torch.sqrt(
        combined.var(
            dim=0,
            unbiased=False
        )
        +
        1e-4
    )


    variance_loss = torch.relu(
        VARIANCE_TARGET_STD
        -
        std
    ).mean()


    total_loss = (

        CONSISTENCY_WEIGHT
        *
        consistency_loss

        +

        VARIANCE_WEIGHT
        *
        variance_loss

    )


    return (
        total_loss,
        consistency_loss,
        variance_loss
    )


# ==================================================================================================
# 21. OPTIMIZER
# ==================================================================================================

trainable_params = [

    p

    for p in model.parameters()

    if p.requires_grad

]


optimizer = torch.optim.AdamW(

    trainable_params,

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY

)


# ==================================================================================================
# 22. LR SCHEDULER
# ==================================================================================================

steps_per_epoch = math.ceil(
    len(
        loader
    )
    /
    GRAD_ACCUM_STEPS
)


total_optimizer_steps = (
    MAX_EPOCHS
    *
    steps_per_epoch
)


warmup_steps = (
    WARMUP_EPOCHS
    *
    steps_per_epoch
)


def lr_lambda(
    current_step
):

    if (
        current_step
        <
        warmup_steps
    ):

        return max(

            1e-8,

            float(
                current_step + 1
            )
            /
            max(
                1,
                warmup_steps
            )

        )


    progress = (

        current_step
        -
        warmup_steps

    ) / max(

        1,

        total_optimizer_steps
        -
        warmup_steps

    )


    progress = min(
        max(
            progress,
            0.0
        ),
        1.0
    )


    cosine = (
        0.5
        *
        (
            1.0
            +
            math.cos(
                math.pi
                *
                progress
            )
        )
    )


    return (

        MIN_LR_RATIO

        +

        (
            1.0
            -
            MIN_LR_RATIO
        )
        *
        cosine

    )


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)


# ==================================================================================================
# 23. AMP
# ==================================================================================================

USE_AMP = (
    DEVICE.type
    ==
    "cuda"
)


try:

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

except Exception:

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )


# ==================================================================================================
# 24. RESUME STATE
# ==================================================================================================

start_epoch = 1

global_optimizer_step = 0

best_epoch = 0

best_loss = float(
    "inf"
)

history = []


if LAST_MODEL_PATH.exists():

    print(
        "\n" +
        "=" * 150
    )

    print(
        "6. RESUME CHECKPOINT FOUND"
    )

    print(
        "=" * 150
    )


    checkpoint = torch.load(
        LAST_MODEL_PATH,
        map_location="cpu",
        weights_only=False
    )


    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=True
    )


    optimizer.load_state_dict(
        checkpoint[
            "optimizer_state_dict"
        ]
    )


    scheduler.load_state_dict(
        checkpoint[
            "scheduler_state_dict"
        ]
    )


    try:

        scaler.load_state_dict(
            checkpoint[
                "scaler_state_dict"
            ]
        )

    except Exception:

        pass


    start_epoch = (
        int(
            checkpoint[
                "epoch"
            ]
        )
        +
        1
    )


    global_optimizer_step = int(
        checkpoint.get(
            "global_optimizer_step",
            0
        )
    )


    best_epoch = int(
        checkpoint.get(
            "best_epoch",
            0
        )
    )


    best_loss = float(
        checkpoint.get(
            "best_loss",
            float(
                "inf"
            )
        )
    )


    history = checkpoint.get(
        "history",
        []
    )


    print(
        "Resume from epoch:",
        start_epoch
    )


    print(
        "Best previous epoch:",
        best_epoch
    )


    print(
        "Best previous loss:",
        best_loss
    )


else:

    print(
        "\n[INFO] No L20C resume checkpoint found."
    )

    print(
        "[INFO] Starting L20C from epoch 1."
    )


# ==================================================================================================
# 25. TRAINING FUNCTION
# ==================================================================================================

def train_one_epoch(
    epoch
):

    global global_optimizer_step


    dataset.set_epoch(
        epoch
    )


    model.train()


    # ----------------------------------------------------------------------------------------------
    # Frozen modules stay frozen by requires_grad.
    # ----------------------------------------------------------------------------------------------

    optimizer.zero_grad(
        set_to_none=True
    )


    running_total = 0.0

    running_consistency = 0.0

    running_variance = 0.0

    sample_count = 0


    epoch_start = time.time()


    for batch_index, batch in enumerate(
        loader,
        start=1
    ):

        view_a = batch[
            "view_a"
        ].to(
            DEVICE,
            non_blocking=True
        )


        view_b = batch[
            "view_b"
        ].to(
            DEVICE,
            non_blocking=True
        )


        # Hugging Face VideoMAE expects:
        #
        # [B, T, C, H, W]
        #
        # Dataset already returns [T,C,H,W].

        with torch.autocast(

            device_type=DEVICE.type,

            dtype=torch.float16
            if DEVICE.type == "cuda"
            else torch.bfloat16,

            enabled=USE_AMP

        ):

            z1 = get_video_representation(
                view_a
            )


            z2 = get_video_representation(
                view_b
            )


            (
                loss,
                consistency_loss,
                variance_loss

            ) = representation_loss(
                z1,
                z2
            )


            scaled_loss = (
                loss
                /
                GRAD_ACCUM_STEPS
            )


        scaler.scale(
            scaled_loss
        ).backward()


        should_step = (

            batch_index
            %
            GRAD_ACCUM_STEPS
            ==
            0

            or

            batch_index
            ==
            len(
                loader
            )

        )


        if should_step:

            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                GRAD_CLIP
            )


            scaler.step(
                optimizer
            )


            scaler.update()


            optimizer.zero_grad(
                set_to_none=True
            )


            scheduler.step()


            global_optimizer_step += 1


        batch_size_actual = view_a.shape[
            0
        ]


        running_total += (
            float(
                loss.detach().item()
            )
            *
            batch_size_actual
        )


        running_consistency += (
            float(
                consistency_loss.detach().item()
            )
            *
            batch_size_actual
        )


        running_variance += (
            float(
                variance_loss.detach().item()
            )
            *
            batch_size_actual
        )


        sample_count += (
            batch_size_actual
        )


        if (
            batch_index
            %
            LOG_EVERY
            ==
            0

            or

            batch_index
            ==
            len(
                loader
            )
        ):

            elapsed = (
                time.time()
                -
                epoch_start
            )


            current_lr = optimizer.param_groups[
                0
            ][
                "lr"
            ]


            print(

                f"Epoch {epoch:02d}/{MAX_EPOCHS:02d} | "
                f"{batch_index:5d}/{len(loader):5d} | "
                f"samples {sample_count:5d}/{len(dataset):5d} | "
                f"Loss {running_total / max(sample_count,1):.5f} | "
                f"Cons {running_consistency / max(sample_count,1):.5f} | "
                f"Var {running_variance / max(sample_count,1):.5f} | "
                f"LR {current_lr:.3e} | "
                f"{elapsed / 60.0:.1f} min"

            )


        del (
            view_a,
            view_b,
            z1,
            z2,
            loss,
            scaled_loss
        )


    epoch_seconds = (
        time.time()
        -
        epoch_start
    )


    return {

        "loss":
            running_total
            /
            max(
                sample_count,
                1
            ),

        "consistency_loss":
            running_consistency
            /
            max(
                sample_count,
                1
            ),

        "variance_loss":
            running_variance
            /
            max(
                sample_count,
                1
            ),

        "samples":
            int(
                sample_count
            ),

        "seconds":
            float(
                epoch_seconds
            ),

        "lr":
            float(
                optimizer.param_groups[
                    0
                ][
                    "lr"
                ]
            )

    }


# ==================================================================================================
# 26. TRAIN
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "7. L20C DOMAIN ADAPTATION"
)

print(
    "=" * 150
)

print()

print(
    "Eligible videos:",
    len(
        dataset
    )
)

print(
    "Excluded validation videos:",
    len(
        validation_df
    )
)

print(
    "Validation decoded during L20C: NO"
)

print(
    "Gloss labels used: NO"
)

print(
    "Epochs:",
    MAX_EPOCHS
)

print(
    "Initial LR:",
    LEARNING_RATE
)

print(
    "Trainable final blocks:",
    TRAINABLE_FINAL_BLOCKS
)

print()


if start_epoch > MAX_EPOCHS:

    print(
        "[INFO] L20C already completed all configured epochs."
    )


for epoch in range(
    start_epoch,
    MAX_EPOCHS + 1
):

    metrics = train_one_epoch(
        epoch
    )


    row = {

        "epoch":
            int(
                epoch
            ),

        "loss":
            float(
                metrics[
                    "loss"
                ]
            ),

        "consistency_loss":
            float(
                metrics[
                    "consistency_loss"
                ]
            ),

        "variance_loss":
            float(
                metrics[
                    "variance_loss"
                ]
            ),

        "learning_rate":
            float(
                metrics[
                    "lr"
                ]
            ),

        "samples":
            int(
                metrics[
                    "samples"
                ]
            ),

        "seconds":
            float(
                metrics[
                    "seconds"
                ]
            ),

        "minutes":
            float(
                metrics[
                    "seconds"
                ]
                /
                60.0
            )

    }


    history.append(
        row
    )


    improved = (
        metrics[
            "loss"
        ]
        <
        best_loss
    )


    if improved:

        best_loss = float(
            metrics[
                "loss"
            ]
        )


        best_epoch = int(
            epoch
        )


        best_checkpoint = {

            "stage":
                "L20C",

            "epoch":
                int(
                    epoch
                ),

            "best_loss":
                float(
                    best_loss
                ),

            "model_state_dict":
                model.state_dict(),

            "base_model":
                BASE_MODEL,

            "trainable_final_blocks":
                TRAINABLE_FINAL_BLOCKS,

            "num_frames":
                NUM_FRAMES,

            "input_size":
                INPUT_SIZE,

            "eligible_videos":
                int(
                    len(
                        dataset
                    )
                ),

            "excluded_validation_videos":
                int(
                    len(
                        validation_df
                    )
                ),

            "validation_used":
                False,

            "gloss_labels_used":
                False

        }


        torch.save(
            best_checkpoint,
            BEST_MODEL_PATH
        )


    # ----------------------------------------------------------------------------------------------
    # Resume checkpoint
    # ----------------------------------------------------------------------------------------------

    last_checkpoint = {

        "stage":
            "L20C",

        "epoch":
            int(
                epoch
            ),

        "global_optimizer_step":
            int(
                global_optimizer_step
            ),

        "best_epoch":
            int(
                best_epoch
            ),

        "best_loss":
            float(
                best_loss
            ),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "scaler_state_dict":
            scaler.state_dict(),

        "history":
            history,

        "base_model":
            BASE_MODEL,

        "eligible_videos":
            int(
                len(
                    dataset
                )
            ),

        "excluded_validation_videos":
            int(
                len(
                    validation_df
                )
            )

    }


    torch.save(
        last_checkpoint,
        LAST_MODEL_PATH
    )


    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )


    print()


    print(
        f"Epoch {epoch:02d} COMPLETE | "
        f"Loss {metrics['loss']:.6f} | "
        f"Consistency {metrics['consistency_loss']:.6f} | "
        f"Variance {metrics['variance_loss']:.6f} | "
        f"{metrics['seconds'] / 60.0:.2f} min | "
        f"{'★ BEST' if improved else ''}"
   )     

    print(
        f"Best epoch: {best_epoch} | "
        f"Best loss: {best_loss:.6f}"
    )

    print(
        "-" * 150
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ==================================================================================================
# 27. LOAD BEST CHECKPOINT
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "8. LOAD BEST L20C CHECKPOINT"
)

print(
    "=" * 150
)


if not BEST_MODEL_PATH.exists():

    raise RuntimeError(
        "Best L20C checkpoint was not created."
    )


best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
    weights_only=False
)


model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ],
    strict=True
)


model = model.to(
    DEVICE
)


best_epoch = int(
    best_checkpoint[
        "epoch"
    ]
)


best_loss = float(
    best_checkpoint[
        "best_loss"
    ]
)


print(
    "Best epoch:",
    best_epoch
)


print(
    "Best adaptation loss:",
    best_loss
)


# ==================================================================================================
# 28. EXPORT ADAPTED BACKBONE
# ==================================================================================================
#
# L20D does not need the Kinetics classifier.
#
# Save only model.videomae state.
#
# ==================================================================================================

backbone_export = {

    "stage":
        "L20C",

    "status":
        "FROZEN_BACKBONE",

    "base_model":
        BASE_MODEL,

    "best_epoch":
        best_epoch,

    "best_adaptation_loss":
        best_loss,

    "num_frames":
        NUM_FRAMES,

    "input_size":
        INPUT_SIZE,

    "trainable_final_blocks":
        TRAINABLE_FINAL_BLOCKS,

    "domain_adaptation_videos":
        int(
            len(
                eligible_df
            )
        ),

    "excluded_validation_videos":
        int(
            len(
                validation_df
            )
        ),

    "validation_used":
        False,

    "gloss_labels_used":
        False,

    "videomae_state_dict":
        model.videomae.state_dict()

}


torch.save(
    backbone_export,
    BACKBONE_PATH
)


print(
    "[PASS] CISLR-adapted VideoMAE backbone exported."
)


# ==================================================================================================
# 29. SUMMARY
# ==================================================================================================

history_df = pd.DataFrame(
    history
)


if len(
    history_df
) == 0:

    raise RuntimeError(
        "Training history is empty."
    )


summary = {

    "stage":
        "L20C",

    "status":
        "COMPLETE",

    "method":
        (
            "validation-excluded label-agnostic "
            "VideoMAE temporal-consistency domain adaptation"
        ),

    "important_methodological_note":
        (
            "This is not original masked-pixel VideoMAE pretraining. "
            "It is temporal-view representation consistency adaptation "
            "using a pretrained VideoMAE backbone."
        ),

    "seed":
        SEED,

    "base_model":
        BASE_MODEL,

    "full_cislr_videos":
        int(
            len(
                full_df
            )
        ),

    "excluded_validation_videos":
        int(
            len(
                validation_df
            )
        ),

    "domain_adaptation_videos":
        int(
            len(
                eligible_df
            )
        ),

    "validation_uid_overlap":
        int(
            len(
                overlap
            )
        ),

    "validation_decoded":
        False,

    "validation_used_for_training":
        False,

    "validation_used_for_model_selection":
        False,

    "gloss_labels_used":
        False,

    "epochs_executed":
        int(
            history_df[
                "epoch"
            ].max()
        ),

    "best_epoch":
        int(
            best_epoch
        ),

    "best_adaptation_loss":
        float(
            best_loss
        ),

    "trainable_final_blocks":
        TRAINABLE_FINAL_BLOCKS,

    "batch_size":
        BATCH_SIZE,

    "gradient_accumulation_steps":
        GRAD_ACCUM_STEPS,

    "effective_batch_size":
        (
            BATCH_SIZE
            *
            GRAD_ACCUM_STEPS
        ),

    "next_stage":
        (
            "L20D_215_CLASS_SUPERVISED_VIDEOMAE"
        ),

    "created_at":
        datetime.now().isoformat()

}


save_json(
    summary,
    SUMMARY_PATH
)


# ==================================================================================================
# 30. HASH ARTIFACTS
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "9. HASH L20C ARTIFACTS"
)

print(
    "=" * 150
)


critical_artifacts = {

    "eligible_manifest":
        ELIGIBLE_MANIFEST_PATH,

    "excluded_validation":
        EXCLUDED_VALIDATION_PATH,

    "config":
        CONFIG_PATH,

    "best_checkpoint":
        BEST_MODEL_PATH,

    "resume_checkpoint":
        LAST_MODEL_PATH,

    "adapted_backbone":
        BACKBONE_PATH,

    "training_history":
        HISTORY_PATH,

    "summary":
        SUMMARY_PATH

}


hash_manifest = {}


for name, path in (
    critical_artifacts.items()
):

    if not path.exists():

        raise FileNotFoundError(
            path
        )


    digest = sha256_file(
        path
    )


    hash_manifest[
        name
    ] = {

        "path":
            str(
                path
            ),

        "sha256":
            digest

    }


    print(
        f"{name:25s}: {digest}"
    )


save_json(
    hash_manifest,
    HASH_PATH
)


# ==================================================================================================
# 31. MASTER FREEZE
# ==================================================================================================

master_freeze = {

    "stage":
        "L20C",

    "status":
        "COMPLETE_AND_FROZEN",

    "seed":
        SEED,

    "method":
        (
            "validation-excluded temporal-consistency "
            "VideoMAE domain adaptation"
        ),

    "base_model":
        BASE_MODEL,

    "full_cislr_population":
        int(
            len(
                full_df
            )
        ),

    "validation_population_excluded":
        int(
            len(
                validation_df
            )
        ),

    "domain_adaptation_population":
        int(
            len(
                eligible_df
            )
        ),

    "validation_uid_overlap":
        int(
            len(
                overlap
            )
        ),

    "validation_video_decoding":
        False,

    "validation_model_selection":
        False,

    "gloss_supervision":
        False,

    "best_epoch":
        int(
            best_epoch
        ),

    "best_adaptation_loss":
        float(
            best_loss
        ),

    "backbone_path":
        str(
            BACKBONE_PATH
        ),

    "next_stage":
        (
            "L20D_215_CLASS_SUPERVISED_TRAINING"
        ),

    "created_at":
        datetime.now().isoformat()

}


save_json(
    master_freeze,
    FREEZE_PATH
)


MASTER_SHA256 = sha256_file(
    FREEZE_PATH
)


# ==================================================================================================
# 32. FINAL INTEGRITY
# ==================================================================================================

print(
    "\n" +
    "=" * 150
)

print(
    "10. FINAL L20C INTEGRITY CHECK"
)

print(
    "=" * 150
)


checks = {

    "full_population_7048":
        len(
            full_df
        )
        ==
        7048,

    "validation_population_307":
        len(
            validation_df
        )
        ==
        307,

    "eligible_population_6741":
        len(
            eligible_df
        )
        ==
        6741,

    "validation_uid_overlap_zero":
        len(
            overlap
        )
        ==
        0,

    "eligible_manifest_exists":
        ELIGIBLE_MANIFEST_PATH.exists(),

    "best_checkpoint_exists":
        BEST_MODEL_PATH.exists(),

    "resume_checkpoint_exists":
        LAST_MODEL_PATH.exists(),

    "adapted_backbone_exists":
        BACKBONE_PATH.exists(),

    "history_exists":
        HISTORY_PATH.exists(),

    "summary_exists":
        SUMMARY_PATH.exists(),

    "validation_decoded":
        False,

    "validation_used_for_training":
        False,

    "validation_used_for_selection":
        False,

    "gloss_labels_used":
        False,

    "freeze_exists":
        FREEZE_PATH.exists()

}


for name, passed in (
    checks.items()
):

    print(
        f"{name:45s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


failed = [

    name

    for name, passed
    in checks.items()

    if not passed

]


if failed:

    raise RuntimeError(
        f"L20C integrity failure: {failed}"
    )


# ==================================================================================================
# 33. FINAL REPORT
# ==================================================================================================

print(
    "\n"
)

print(
    "=" * 150
)

print(
    "L20C — FULL CISLR VIDEOMAE DOMAIN ADAPTATION COMPLETE"
)

print(
    "=" * 150
)

print()


print(
    "DATA"
)

print(
    f"  Full clean CISLR videos       : {len(full_df)}"
)

print(
    f"  Validation videos excluded    : {len(validation_df)}"
)

print(
    f"  Domain-adaptation videos      : {len(eligible_df)}"
)

print(
    f"  Validation UID overlap        : {len(overlap)}"
)


print()


print(
    "MODEL"
)

print(
    f"  Base                          : {BASE_MODEL}"
)

print(
    f"  Trainable final blocks        : {TRAINABLE_FINAL_BLOCKS}"
)

print(
    f"  Frames / video view           : {NUM_FRAMES}"
)

print(
    f"  Input size                    : {INPUT_SIZE}x{INPUT_SIZE}"
)


print()


print(
    "TRAINING"
)

print(
    f"  Epochs                        : {int(history_df['epoch'].max())}"
)

print(
    f"  Batch size                    : {BATCH_SIZE}"
)

print(
    f"  Gradient accumulation         : {GRAD_ACCUM_STEPS}"
)

print(
    f"  Effective batch size          : {BATCH_SIZE * GRAD_ACCUM_STEPS}"
)

print(
    f"  Initial LR                    : {LEARNING_RATE:.2e}"
)


print()


print(
    "BEST DOMAIN-ADAPTATION CHECKPOINT"
)

print(
    f"  Best epoch                    : {best_epoch}"
)

print(
    f"  Best adaptation loss          : {best_loss:.6f}"
)


print()


print(
    "LEAKAGE PROTECTION"
)

print(
    "  Validation videos decoded     : NO"
)

print(
    "  Validation used for training  : NO"
)

print(
    "  Validation used for selection : NO"
)

print(
    "  Gloss labels used             : NO"
)


print()


print(
    "ARTIFACTS"
)

print(
    "  Adapted backbone:"
)

print(
    f"    {BACKBONE_PATH}"
)

print()

print(
    "  Best checkpoint:"
)

print(
    f"    {BEST_MODEL_PATH}"
)

print()

print(
    "  Training history:"
)

print(
    f"    {HISTORY_PATH}"
)

print()

print(
    "  Master freeze:"
)

print(
    f"    {FREEZE_PATH}"
)


print()


print(
    "L20C MASTER SHA256:"
)

print(
    f"  {MASTER_SHA256}"
)


print()


print(
    "[PASS] L20C COMPLETE AND FROZEN"
)

print(
    "[PASS] 6741 CISLR VIDEOS USED FOR DOMAIN ADAPTATION"
)

print(
    "[PASS] ALL 307 VALIDATION VIDEOS REMAINED UNSEEN"
)

print(
    "[PASS] ZERO VALIDATION UID OVERLAP"
)

print(
    "[PASS] NO GLOSS LABELS USED DURING L20C"
)

print(
    "[PASS] CISLR-ADAPTED VIDEOMAE BACKBONE EXPORTED"
)

print()

print(
    "[NEXT] L20D = 215-CLASS SUPERVISED VIDEOMAE TRAINING"
)

print(
    "=" * 150
)

CISLR — L20C
VALIDATION-EXCLUDED VIDEOMAE DOMAIN ADAPTATION

Python: 3.11.15
PyTorch: 2.14.0+cu130
OpenCV: 5.0.0
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU memory: 5.64 GB

1. VERIFY FROZEN L20B
full manifest                 : PASS
train manifest                : PASS
validation manifest           : PASS
L20B freeze                   : PASS
[PASS] L20B freeze verified.

2. BUILD VALIDATION-EXCLUDED L20C POPULATION
Full clean population: 7048
L20B supervised train: 717
Frozen validation: 307

L20C eligible videos: 6741
Expected: 6741
L20C ↔ validation UID overlap: 0
[PASS] 307 validation videos excluded from L20C.

3. VERIFY L20C VIDEO PATHS
Resolved: 6741
Missing: 0

4. DATA LOADER
Dataset videos: 6741
Batches/epoch: 6741
Batch size: 1
Gradient accumulation: 8
Effective batch size: 8

5. LOAD PRETRAINED VIDEOMAE
Base model loaded: MCG-NJU/videomae-base-finetuned-kinetics
Transformer blocks: 12
[INFO] model.videomae.layernorm = None
[INFO] Skipping final layernorm unfreeze

RuntimeError: L20C integrity failure: ['validation_decoded', 'validation_used_for_training', 'validation_used_for_selection', 'gloss_labels_used']

In [4]:
# ==============================================================================================
# L20C — STANDALONE FINAL INTEGRITY CHECK
# NO TRAINING — NO MODEL INFERENCE — SAFE TO RUN AFTER L20C
# ==============================================================================================

from pathlib import Path
import pandas as pd
import json

print("=" * 120)
print("L20C — STANDALONE FINAL INTEGRITY CHECK")
print("=" * 120)

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL"
)

L20B_DIR = ROOT / "audit" / "l20b_full_cislr_protocol_freeze"
L20C_DIR = ROOT / "audit" / "l20c_full_cislr_domain_adaptation"

print("\nL20B directory:", L20B_DIR)
print("L20C directory:", L20C_DIR)

assert L20B_DIR.exists(), f"L20B directory not found: {L20B_DIR}"
assert L20C_DIR.exists(), f"L20C directory not found: {L20C_DIR}"


# ==============================================================================================
# Helper: find CSV based on expected row count + filename hints
# ==============================================================================================

def find_csv(base_dir, expected_rows=None, hints=()):

    candidates = list(base_dir.rglob("*.csv"))

    matches = []

    for path in candidates:

        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        if expected_rows is not None and len(df) != expected_rows:
            continue

        name = path.name.lower()

        score = sum(
            1 for hint in hints
            if hint.lower() in name
        )

        matches.append(
            (score, path, df)
        )

    if not matches:
        return None, None

    matches.sort(
        key=lambda x: (
            x[0],
            str(x[1])
        ),
        reverse=True
    )

    _, path, df = matches[0]

    return path, df


# ==============================================================================================
# 1. LOAD FULL 7048 MANIFEST
# ==============================================================================================

full_path, full_df = find_csv(
    L20B_DIR,
    expected_rows=7048,
    hints=("full", "manifest")
)

if full_df is None:
    raise RuntimeError(
        "Could not locate the 7048-row L20B full manifest."
    )


# ==============================================================================================
# 2. LOAD FROZEN 307 VALIDATION MANIFEST
# ==============================================================================================

val_path, val_df = find_csv(
    L20B_DIR,
    expected_rows=307,
    hints=("validation", "val", "manifest")
)

if val_df is None:
    raise RuntimeError(
        "Could not locate the 307-row L20B validation manifest."
    )


# ==============================================================================================
# 3. LOAD L20C 6741 ELIGIBLE MANIFEST
# ==============================================================================================

eligible_path, eligible_df = find_csv(
    L20C_DIR,
    expected_rows=6741,
    hints=("eligible", "manifest")
)

if eligible_df is None:
    raise RuntimeError(
        "Could not locate the 6741-row L20C eligible manifest."
    )


print("\nFOUND MANIFESTS")
print("-" * 120)

print("Full manifest      :", full_path)
print("Validation manifest:", val_path)
print("Eligible manifest  :", eligible_path)

print()

print("Full rows      :", len(full_df))
print("Validation rows:", len(val_df))
print("Eligible rows  :", len(eligible_df))


# ==============================================================================================
# 4. DETECT UID COLUMN
# ==============================================================================================

def detect_uid_column(df):

    possibilities = [
        "uid",
        "UID",
        "video_uid",
        "video_id",
        "id"
    ]

    for col in possibilities:

        if col in df.columns:
            return col

    raise RuntimeError(
        f"Could not determine UID column. Columns = {list(df.columns)}"
    )


full_uid_col = detect_uid_column(full_df)
val_uid_col = detect_uid_column(val_df)
eligible_uid_col = detect_uid_column(eligible_df)

print("\nUID columns")
print("-" * 120)

print("Full      :", full_uid_col)
print("Validation:", val_uid_col)
print("Eligible  :", eligible_uid_col)


# ==============================================================================================
# 5. BUILD UID SETS
# ==============================================================================================

full_uids = set(
    full_df[full_uid_col]
    .astype(str)
    .str.strip()
)

val_uids = set(
    val_df[val_uid_col]
    .astype(str)
    .str.strip()
)

eligible_uids = set(
    eligible_df[eligible_uid_col]
    .astype(str)
    .str.strip()
)


overlap = eligible_uids & val_uids

union = eligible_uids | val_uids


# ==============================================================================================
# 6. POSITIVE INTEGRITY CHECKS
# ==============================================================================================

checks = {

    "full_population_7048":
        len(full_df) == 7048,

    "validation_population_307":
        len(val_df) == 307,

    "eligible_population_6741":
        len(eligible_df) == 6741,

    "full_unique_uids_7048":
        len(full_uids) == 7048,

    "validation_unique_uids_307":
        len(val_uids) == 307,

    "eligible_unique_uids_6741":
        len(eligible_uids) == 6741,

    "validation_uid_overlap_zero":
        len(overlap) == 0,

    "eligible_plus_validation_equals_full":
        union == full_uids,

    "population_accounting":
        len(eligible_df) + len(val_df) == len(full_df),
}


print()
print("=" * 120)
print("DATA INTEGRITY")
print("=" * 120)

failed = []

for name, passed in checks.items():

    print(
        f"{name:<50}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:
        failed.append(name)


print()
print("Validation/eligible UID overlap:", len(overlap))


# ==============================================================================================
# 7. VERIFY SAVED MODEL ARTIFACTS
# ==============================================================================================

print()
print("=" * 120)
print("MODEL ARTIFACTS")
print("=" * 120)

all_files = [
    p
    for p in L20C_DIR.rglob("*")
    if p.is_file()
]

checkpoint_files = [
    p for p in all_files
    if p.suffix.lower() in {".pt", ".pth", ".bin"}
]

json_files = [
    p for p in all_files
    if p.suffix.lower() == ".json"
]

csv_files = [
    p for p in all_files
    if p.suffix.lower() == ".csv"
]


print("Checkpoint/model files:", len(checkpoint_files))

for p in checkpoint_files:
    print("  ", p)


print("\nJSON files:", len(json_files))

for p in json_files:
    print("  ", p)


print("\nCSV files:", len(csv_files))

for p in csv_files:
    print("  ", p)


artifact_check = len(checkpoint_files) > 0

print(
    "\nmodel_checkpoint_exists".ljust(50),
    ":",
    "PASS" if artifact_check else "FAIL"
)

if not artifact_check:
    failed.append("model_checkpoint_exists")


# ==============================================================================================
# 8. FORBIDDEN EVENTS
#
# These values describe the actual L20C protocol.
# False is GOOD because these events must NOT occur.
# ==============================================================================================

forbidden_events = {

    "validation_decoded":
        False,

    "validation_used_for_training":
        False,

    "validation_used_for_selection":
        False,

    "gloss_labels_used":
        False,
}


print()
print("=" * 120)
print("FORBIDDEN EVENTS")
print("=" * 120)

for name, occurred in forbidden_events.items():

    passed = not occurred

    print(
        f"{name:<50}: "
        f"{'PASS' if passed else 'FAIL'} "
        f"(occurred={occurred})"
    )

    if occurred:
        failed.append(name)


# ==============================================================================================
# 9. FINAL RESULT
# ==============================================================================================

print()
print("=" * 120)
print("FINAL RESULT")
print("=" * 120)

if failed:

    print("[FAIL] L20C integrity verification failed.")
    print("Failed checks:")

    for item in failed:
        print(" -", item)

else:

    print("[PASS] L20C DATA INTEGRITY VERIFIED.")
    print()
    print(f"Full population       : {len(full_df)}")
    print(f"L20C eligible         : {len(eligible_df)}")
    print(f"Frozen validation     : {len(val_df)}")
    print(f"UID overlap           : {len(overlap)}")
    print()
    print("Validation decoded             : NO")
    print("Validation used for training   : NO")
    print("Validation used for selection  : NO")
    print("Gloss labels used in L20C      : NO")
    print()
    print("[PASS] L20C training does NOT need to be rerun.")
    print("[PASS] Existing best L20C checkpoint/backbone can be retained.")

print("=" * 120)

L20C — STANDALONE FINAL INTEGRITY CHECK

L20B directory: /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_full_cislr_protocol_freeze
L20C directory: /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20c_full_cislr_domain_adaptation

FOUND MANIFESTS
------------------------------------------------------------------------------------------------------------------------
Full manifest      : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_full_cislr_protocol_freeze/manifests/l20b_full_7048_domain_pretraining_manifest.csv
Validation manifest: /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_full_cislr_protocol_freeze/manifests/l20b_validation_manifest.csv
Eligible manifest  : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20c_full_cislr_domain_adaptation/manifests/l20c_domain_adaptation_6741_manifest.csv

Full rows      : 7048
Validation rows: 307
Eligible rows  : 6741

UID colu

In [5]:
from pathlib import Path
import pandas as pd

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL"
)

# L19 locked test
L19_TEST = (
    ROOT
    / "audit"
    / "l18b_10class_dataset"
    / "manifests"
    / "l18b_LOCKED_test_manifest.csv"
)

# L20C videos actually used
L20C_ELIGIBLE = (
    ROOT
    / "audit"
    / "l20c_full_cislr_domain_adaptation"
    / "manifests"
    / "l20c_domain_adaptation_6741_manifest.csv"
)

assert L19_TEST.exists(), L19_TEST
assert L20C_ELIGIBLE.exists(), L20C_ELIGIBLE

test_df = pd.read_csv(L19_TEST)
l20c_df = pd.read_csv(L20C_ELIGIBLE)

test_uids = set(test_df["uid"].astype(str).str.strip())
l20c_uids = set(l20c_df["uid"].astype(str).str.strip())

overlap = sorted(test_uids & l20c_uids)

print("=" * 90)
print("L19 LOCKED TEST vs L20C TRAINING OVERLAP CHECK")
print("=" * 90)

print("L19 locked test videos :", len(test_uids))
print("L20C training videos   :", len(l20c_uids))
print("Overlap                :", len(overlap))

if overlap:
    print("\n[FAIL] Locked L19 test videos were exposed during L20C.")
    print("\nOverlapping UIDs:")
    for uid in overlap:
        print(" ", uid)

    print("\nDO NOT START L20D FROM THIS L20C BACKBONE.")

else:
    print("\n[PASS] Zero overlap.")
    print("[PASS] L19 locked test remained untouched by L20C.")
    print("[PASS] We can proceed toward L20D.")

print("=" * 90)

L19 LOCKED TEST vs L20C TRAINING OVERLAP CHECK
L19 locked test videos : 10
L20C training videos   : 6741
Overlap                : 6

[FAIL] Locked L19 test videos were exposed during L20C.

Overlapping UIDs:
  5pYCIsZH-H4
  6gXuMMAxOm0
  7R_b_tfp5lM
  DfohDPZVlrA_1
  IOS2_Xy4yg0
  SojDDNUsSew_1

DO NOT START L20D FROM THIS L20C BACKBONE.
